## Gold Cascade Modeling (Deliverable 1.3.3)

This notebook implements the full three-stage cascade anomaly detection pipeline. The objective is to determine whether a layered approach improves alert quality when compared to the single-model baseline.

**Purpose:**  
To operationalize the cascade architecture defined in Section C, generating alert outputs for each stage of the cascade: broad detection (Stage 1), refined detection (Stage 2), and rule/profile/historical confirmation (Stage 3).

**Stages Implemented:**

1. **Stage 1 — Broad Isolation Forest**  
   High-sensitivity anomaly screen using all 50 Gold numeric features.

2. **Stage 2 — Narrow Isolation Forest**  
   Secondary detector trained on the reduced feature subset identified during Silver EDA.

3. **Stage 3 — Rule / Profile / Historical Confirmation**  
   Final confirmation based on behavior profiles, persistence checks, drift features, and cross-sensor consistency.

**Key Goals:**

- Load the Gold preprocessed dataset and Gold feature artifacts.
- Train Stage 1 and Stage 2 Isolation Forest models.
- Apply Stage 3 rule/profile confirmation logic based on Silver EDA outputs.
- Generate and store alert outputs for all three cascade stages.
- Export all cascade artifacts for comparative evaluation.

**Relevance to Section C:**  
This notebook produces the layered alert outputs required for evaluating the cascade’s effect on false positives, noisy alerts, and anomaly sensitivity. These outputs are necessary for the statistical tests, alert-volume comparisons, and visual communication described in Section C.

## Tuned Cascade Modeling Setup and Imports

In this section I am loading the libraries and project utilities needed for the tuned Gold cascade modeling stage.

The purpose here is to get the notebook ready before I start working with the Gold modeling inputs. That includes:
- standard Python libraries
- model and metrics utilities
- path and config loading
- logging
- truth-record helpers
- artifact saving utilities
- experiment tracking support

At this point I am not fitting the cascade yet. I am just preparing the notebook so the tuned multi-stage workflow can run in a structured and repeatable way.

In [57]:
from __future__ import annotations

from dataclasses import dataclass, field
from datetime import datetime, timezone
from typing import Any, Dict, List, Optional, Sequence, Tuple, Union

from pathlib import Path
import yaml

import logging
import wandb

import pandas as pd 
import numpy as np 

import joblib 

from sklearn.model_selection import train_test_split, KFold, ParameterGrid
from sklearn.preprocessing import StandardScaler, MinMaxScaler, OneHotEncoder, RobustScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.ensemble import RandomForestClassifier, IsolationForest
from sklearn.metrics import classification_report, confusion_matrix, precision_recall_fscore_support, roc_auc_score, average_precision_score
from sklearn.svm import OneClassSVM
from sklearn.neighbors import LocalOutlierFactor

# Custom Utilities Module
from utils.paths import get_paths
from utils.file_io import load_data, save_data, save_json, load_json
from utils.eda_logging import profile_dataframe
from utils.logging_setup import configure_logging, log_layer_paths
from utils.wandb_utils import finalize_wandb_stage

from utils.truths import (
    make_process_run_id,
    build_file_fingerprint,
    extract_truth_hash,
    identify_meta_columns,
    identify_feature_columns,
    initialize_layer_truth,
    update_truth_section,
    build_truth_record,
    save_truth_record,
    append_truth_index,
    stamp_truth_columns,
    load_truth_record,
    find_truth_record_by_hash,
    load_truth_record_by_hash,
    load_parent_truth_record_from_dataframe,
    get_truth_value,
    get_dataset_name_from_truth,
    get_truth_hash,
    get_parent_truth_hash,
    get_pipeline_mode_from_truth,
    get_artifact_path_from_truth,
)

from utils.pipeline_config_loader import (
    load_pipeline_config,
    build_truth_config_block,
    set_wandb_dir_from_config,
    export_config_snapshot,
)

from utils.postgres_util import get_engine_from_env
from utils.layer_postgres_writer import write_layer_dataframe, prepare_layer_dataframe



# Ledger 
from utils.ledger import Ledger

# Show more columns
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 200)


----

## Load Configuration, Paths, and Tuned Cascade Runtime Settings

Here I load the resolved configuration values that control how the tuned cascade notebook runs.

This step defines the key runtime pieces for the cascade stage, such as:
- dataset identity
- Gold version and recipe information
- Stage 1 settings
- Stage 2 selection mode
- Stage 2 search grids and minimum recall rule
- Stage 3 rule thresholds
- file names and artifact paths
- truth-store locations
- model output locations

The important difference in this notebook is that the cascade variant is **tuned**. That means Stage 2 is not locked to one fixed setup. Instead, this notebook is allowed to search across candidate parameter and threshold combinations and keep the best one based on the configured selection logic.

In [58]:
paths = get_paths()

#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### 

CONFIG_ROOT = paths.configs
CONFIG_RUN_MODE = "train"
CONFIG_PROFILE = "default"

CASCADE_VARIANT = "stage3_improved"

CONFIG = load_pipeline_config(
    config_root=CONFIG_ROOT,
    stage="gold_cascade",
    dataset="pump",
    mode=CONFIG_RUN_MODE,
    profile=CONFIG_PROFILE,
    project_root=paths.root,
).data

GOLD_CFG = CONFIG["gold_cascade"]

PATHS = CONFIG["resolved_paths"]
FILENAMES = CONFIG["filenames"]
PIPELINE = CONFIG.get(
    "pipeline",
    {
        "execution_mode": "batch",
        "orchestration_mode": "notebook",
    },
)

TRUTH_CONFIG = build_truth_config_block(CONFIG)
TRUTH_CONFIG["pipeline"] = PIPELINE

# Stage details
STAGE = "gold"
LAYER_NAME = GOLD_CFG["layer_name"]
GOLD_VERSION = CONFIG["versions"]["gold"]
RECIPE_ID = GOLD_CFG["recipe_id"]
TRUTH_VERSION = CONFIG["versions"]["truth"]
PIPELINE_MODE = PIPELINE["execution_mode"]
RUN_MODE = CONFIG["runtime"]["mode"]

DATASET_NAME_CONFIG = CONFIG["dataset"]["name"]
DATASET_NAME = str(DATASET_NAME_CONFIG).strip().lower()

GOLD_PROCESS_RUN_ID = make_process_run_id(GOLD_CFG["process_run_id_prefix"])

#### #### #### #### 

STAGE1_CFG = GOLD_CFG["stage1"]
STAGE2_CFG = GOLD_CFG["stage2"]
STAGE3_CFG = GOLD_CFG["stage3"]

RANDOM_SEED = int(GOLD_CFG["random_seed"])

STAGE1_ESTIMATOR_COUNT = int(GOLD_CFG["stage1"]["estimator_count"])
STAGE1_THRESHOLD_PERCENTILE = float(GOLD_CFG["stage1"]["threshold_percentile"])

# ---------------------------------------------------------
# Stage 2
# ---------------------------------------------------------
STAGE2_SELECTION_MODE = "fixed"
STAGE2_RANDOM_STATE = int(STAGE2_CFG.get("random_state", RANDOM_SEED))
STAGE2_MIN_RECALL = float(STAGE2_CFG["min_recall"])

STAGE2_FIXED_PARAMS = {
    # example:
    # "n_estimators": 300,
    # "max_samples": 256,
    # "contamination": "auto",
    # "max_features": 1.0,
}
STAGE2_FIXED_THRESHOLD_PERCENTILE = float(0.95)  

STAGE2_THRESHOLD_GRID = [
    float(value)
    for value in STAGE2_CFG["search"]["threshold_grid"]
]

STAGE2_PARAM_GRID = {
    key: list(value)
    for key, value in STAGE2_CFG["search"]["param_grid"].items()
}

# ---------------------------------------------------------
# Improved Stage 3 settings
# ---------------------------------------------------------
STAGE3_STRONG_PRIMARY_HITS = 3
STAGE3_MIN_PRIMARY_SENSOR_HITS = 2
STAGE3_MIN_SECONDARY_SENSOR_HITS = 1

STAGE3_PROFILE_BREACH_WEIGHT = 2.0
STAGE3_CORROBORATION_WEIGHT = 1.0
STAGE3_PERSISTENCE_WEIGHT = 1.0
STAGE3_DRIFT_WEIGHT = 1.0
STAGE3_MIN_WEIGHTED_SCORE = 3.0

STAGE3_ROLLING_WINDOW_SIZE = 3
STAGE3_MINIMUM_FLAGS_IN_WINDOW = 2

STAGE3_DRIFT_ROLLING_WINDOW_SIZE = 5
STAGE3_DRIFT_THRESHOLD_MULTIPLIER = 1.0

#### #### #### #### 

# Weights and Biases
WANDB_PROJECT = CONFIG["wandb"]["project"]
WANDB_ENTITY = CONFIG["wandb"]["entity"]
WANDB_RUN_NAME = f"{GOLD_VERSION}"

# File names
GOLD_PREPROCESSED_FILE_NAME = FILENAMES["gold_preprocessed_file_name"]
GOLD_PREPROCESSED_SCALED_FILE_NAME = FILENAMES["gold_preprocessed_scaled_file_name"]

GOLD_FIT_FILE_NAME = FILENAMES["gold_fit_file_name"]
GOLD_TRAIN_FILE_NAME = FILENAMES["gold_train_file_name"]
GOLD_TEST_FILE_NAME = FILENAMES["gold_test_file_name"]

STAGE1_FEATURES_FILE_NAME = FILENAMES["stage1_features_file_name"]
STAGE2_FEATURES_FILE_NAME = FILENAMES["stage2_features_file_name"]
STAGE3_PRIMARY_FILE_NAME = FILENAMES["stage3_primary_file_name"]
STAGE3_SECONDARY_FILE_NAME = FILENAMES["stage3_secondary_file_name"]

STAGE1_MODEL_FILE_NAME = FILENAMES["cascade_stage3_improved_stage1_model_file_name"]
STAGE2_MODEL_FILE_NAME = FILENAMES["cascade_stage3_improved_stage2_model_file_name"]

CASCADE_RESULTS_FILE_NAME_CSV = FILENAMES["cascade_stage3_improved_results_file_name_csv"]
CASCADE_RESULTS_FILE_NAME_PICKLE = FILENAMES["cascade_stage3_improved_results_file_name_pickle"]
CASCADE_THRESHOLDS_FILE_NAME = FILENAMES["cascade_stage3_improved_thresholds_file_name"]
CASCADE_SUMMARY_FILE_NAME = FILENAMES["cascade_stage3_improved_summary_file_name"]
CASCADE_METADATA_FILE_NAME = FILENAMES["cascade_stage3_improved_metadata_file_name"]

CASCADE_TUNED_THRESHOLDS_FILE_NAME = FILENAMES["cascade_tuned_thresholds_file_name"]
CASCADE_TUNED_SUMMARY_FILE_NAME = FILENAMES["cascade_tuned_summary_file_name"]


GOLD_ARTIFACTS_PATH = Path(PATHS["gold_artifacts_dir"])

GOLD_PREPROCESSED_DATA_PATH = Path(PATHS["gold_preprocessed_data_path"])
GOLD_PREPROCESSED_SCALED_DATA_PATH = Path(PATHS["gold_preprocessed_scaled_data_path"])

GOLD_FIT_DATA_PATH = Path(PATHS["gold_fit_data_path"])
GOLD_TRAIN_DATA_PATH = Path(PATHS["gold_train_data_path"])
GOLD_TEST_DATA_PATH = Path(PATHS["gold_test_data_path"])

STAGE1_FEATURES_PATH = Path(PATHS["stage1_features_path"])
STAGE2_FEATURES_PATH = Path(PATHS["stage2_features_path"])
STAGE3_PRIMARY_PATH = Path(PATHS["stage3_primary_path"])
STAGE3_SECONDARY_PATH = Path(PATHS["stage3_secondary_path"])



CASCADE_RESULTS_PATH_CSV = Path(PATHS["cascade_stage3_improved_results_path_csv"])
CASCADE_RESULTS_PATH_PICKLE = Path(PATHS["cascade_stage3_improved_results_path_pickle"])
CASCADE_THRESHOLDS_PATH = Path(PATHS["cascade_stage3_improved_thresholds_path"])
CASCADE_SUMMARY_PATH = Path(PATHS["cascade_stage3_improved_summary_path"])
CASCADE_METADATA_PATH = Path(PATHS["cascade_stage3_improved_metadata_path"])


CASCADE_TUNED_THRESHOLDS_PATH = Path(PATHS["cascade_tuned_thresholds_path"])
CASCADE_TUNED_SUMMARY_PATH = Path(PATHS["cascade_tuned_summary_path"])

GOLD_CASCADE_STAGE3_IMPROVED_LEDGER_FILE_NAME = FILENAMES["gold_cascade_stage3_imporved_ledger_file_name"]
GOLD_LEDGER_PATH = GOLD_ARTIFACTS_PATH / GOLD_CASCADE_STAGE3_IMPROVED_LEDGER_FILE_NAME

set_wandb_dir_from_config(CONFIG)

MODELS_PATH = Path(PATHS["models_root"])

STAGE1_MODELS_PATH = Path(PATHS["cascade_stage3_improved_stage1_models_path"])
STAGE1_MODEL_ARTIFACT_PATH = Path(PATHS["cascade_stage3_improved_stage1_model_artifact_path"])

STAGE2_MODELS_PATH = Path(PATHS["cascade_stage3_improved_stage2_models_path"])
STAGE2_MODEL_ARTIFACT_PATH = Path(PATHS["cascade_stage3_improved_stage2_model_artifact_path"])



CASCADE_REFERENCE_PROFILE_PATH = Path(PATHS["cascade_tuned_reference_profile_path"])


# Logs
LOGS_PATH = Path(PATHS["logs_root"])

# Truths
TRUTHS_PATH = Path(PATHS["truths_dir"])
TRUTH_INDEX_PATH = Path(PATHS["truth_index_path"])

# Path Failsafes

GOLD_PREPROCESSED_DATA_PATH.parent.mkdir(parents=True, exist_ok=True)
GOLD_PREPROCESSED_SCALED_DATA_PATH.parent.mkdir(parents=True, exist_ok=True)
GOLD_FIT_DATA_PATH.parent.mkdir(parents=True, exist_ok=True)
GOLD_TEST_DATA_PATH.parent.mkdir(parents=True, exist_ok=True)
GOLD_TRAIN_DATA_PATH.parent.mkdir(parents=True, exist_ok=True)
GOLD_ARTIFACTS_PATH.mkdir(parents=True, exist_ok=True)
MODELS_PATH.mkdir(parents=True, exist_ok=True)
TRUTHS_PATH.mkdir(parents=True, exist_ok=True)
LOGS_PATH.mkdir(parents=True, exist_ok=True)


----

## Start Logging for the Tuned Cascade Modeling Stage

Before I load the modeling inputs, I want logging turned on so this notebook records what happened during the run.

This helps with debugging, traceability, and basic pipeline discipline. If I need to check what inputs were used or where a step failed, the log gives me a cleaner record than notebook output alone.

In [59]:
# Logging Setup

# Create gold log path 
gold_log_path = paths.logs / "gold_modeling_cascade_stage3_improved.log"

# Initial Logger
configure_logging(
    "capstone",
    gold_log_path,
    level=logging.DEBUG,
    overwrite_handlers=True,
)

# Initiate Logger and log file
logger = logging.getLogger("capstone.gold")

# Log load and initiation
logger.info("Gold Modeling stage starting")

# Log paths loads
log_layer_paths(paths, current_layer="gold", logger=logger)


2026-03-23 23:43:37,437 | INFO | capstone.gold | Gold Modeling stage starting
2026-03-23 23:43:37,441 | INFO | capstone.gold | Project Root Path Loaded: /workspace
2026-03-23 23:43:37,443 | INFO | capstone.gold | Project Logging Path Loaded: /workspace/logs
2026-03-23 23:43:37,445 | INFO | capstone.gold | Project Artifacts Path Loaded: /workspace/artifacts
2026-03-23 23:43:37,446 | INFO | capstone.gold | Project Notebooks Path Loaded: /workspace/notebooks
2026-03-23 23:43:37,447 | INFO | capstone.gold | Project Truths Path Loaded: /workspace/artifacts/truths
2026-03-23 23:43:37,448 | INFO | capstone.gold | Project Data Path Loaded: /workspace/data
2026-03-23 23:43:37,449 | INFO | capstone.gold | Previous Layer (Silver) Path Loaded: /workspace/data/silver
2026-03-23 23:43:37,450 | INFO | capstone.gold | Previous Layer (Silver) Training Path Loaded: /workspace/data/silver/train
2026-03-23 23:43:37,452 | INFO | capstone.gold | Previous Layer (Silver) Testing Path Loaded: /workspace/data/s

----

## Initialize Experiment Tracking

This step starts the Weights & Biases run for the tuned cascade modeling stage.

I am using this mainly for run tracking and artifact registration. It helps document the cascade configuration, inputs, and saved outputs, but it does not change the underlying cascade logic itself.

In [60]:
# W&B

wandb_run = wandb.init(
    project=WANDB_PROJECT,
    entity=WANDB_ENTITY,
    name=WANDB_RUN_NAME,
    job_type="gold_modeling_cascade",
    config={
        "gold_version": GOLD_VERSION,
        "dataset": DATASET_NAME,
        "stage": STAGE,
        "cascade_variant": CASCADE_VARIANT,
        "stage1_threshold_percentile": STAGE1_THRESHOLD_PERCENTILE,
        "stage2_selection_mode": STAGE2_SELECTION_MODE,
        "stage2_fixed_params": STAGE2_FIXED_PARAMS,
        "stage2_fixed_threshold_percentile": STAGE2_FIXED_THRESHOLD_PERCENTILE,
        "stage3_strong_primary_hits": STAGE3_STRONG_PRIMARY_HITS,
        "stage3_min_primary_sensor_hits": STAGE3_MIN_PRIMARY_SENSOR_HITS,
        "stage3_min_secondary_sensor_hits": STAGE3_MIN_SECONDARY_SENSOR_HITS,
        "stage3_profile_breach_weight": STAGE3_PROFILE_BREACH_WEIGHT,
        "stage3_corroboration_weight": STAGE3_CORROBORATION_WEIGHT,
        "stage3_persistence_weight": STAGE3_PERSISTENCE_WEIGHT,
        "stage3_drift_weight": STAGE3_DRIFT_WEIGHT,
        "stage3_min_weighted_score": STAGE3_MIN_WEIGHTED_SCORE,
        "stage3_rolling_window_size": STAGE3_ROLLING_WINDOW_SIZE,
        "stage3_minimum_flags_in_window": STAGE3_MINIMUM_FLAGS_IN_WINDOW,
        "stage3_drift_rolling_window_size": STAGE3_DRIFT_ROLLING_WINDOW_SIZE,
        "stage3_drift_threshold_multiplier": STAGE3_DRIFT_THRESHOLD_MULTIPLIER,
        "gold_fit_path": str(GOLD_FIT_DATA_PATH),
        "gold_scored_path": str(GOLD_PREPROCESSED_SCALED_DATA_PATH),
        "stage1_features_path": str(STAGE1_FEATURES_PATH),
        "stage2_features_path": str(STAGE2_FEATURES_PATH),
        "stage3_primary_path": str(STAGE3_PRIMARY_PATH),
        "stage3_secondary_path": str(STAGE3_SECONDARY_PATH),
    },
)
logger.info("W&B initialized: %s", wandb.run.name)

2026-03-23 23:43:38,999 | INFO | capstone.gold | W&B initialized: gold__001


----

## Initialize the Cascade Ledger

Here I create the ledger that tracks the main steps taken during the tuned cascade notebook.

I treat the ledger as a structured record of the run. It gives me a cleaner summary of the workflow than relying only on printed notebook output, especially when I need to review or compare cascade runs later.

In [61]:
# Ledger Setup

ledger = Ledger(stage=STAGE, recipe_id=RECIPE_ID)

ledger.add(
    kind="step",
    step="init",
    message="Initialized ledger",
    logger=logger
)


2026-03-23 23:43:39,402 | INFO | capstone.gold | LEDGER | {'ts_utc': '2026-03-23T23:43:39.402500+00:00', 'stage': 'gold', 'recipe': 'gold_modeling__v001_cascade', 'kind': 'step', 'step': 'init', 'message': 'Initialized ledger', 'why': None, 'consequence': None, 'data': {}}


{'ts_utc': '2026-03-23T23:43:39.402500+00:00',
 'stage': 'gold',
 'recipe': 'gold_modeling__v001_cascade',
 'kind': 'step',
 'step': 'init',
 'message': 'Initialized ledger',
 'why': None,
 'consequence': None,
 'data': {}}

----

## Load the Gold Modeling Inputs and Resolve the Parent Truth

This is the point where I load the Gold preprocessing outputs that the tuned cascade depends on.

In this step I am:
- loading the scaled Gold dataframe
- resolving the parent Gold truth record
- confirming the dataset identity
- updating artifact paths from the truth record when available
- loading the Gold fit, train, and test dataframes
- loading the Stage 1, Stage 2, and Stage 3 feature artifacts

This matters because the tuned cascade notebook should inherit the prepared Gold artifacts rather than rebuilding those inputs on its own.

In [62]:
logger.info("Loading Gold Preprocessed parquet: %s", GOLD_PREPROCESSED_SCALED_DATA_PATH)
gold_preprocessed_scaled_dataframe = load_data(GOLD_PREPROCESSED_SCALED_DATA_PATH)

GOLD_DATASET_NAME = (
    gold_preprocessed_scaled_dataframe["meta__dataset"]
    .dropna()
    .astype("string")
    .str.strip()
)
GOLD_DATASET_NAME = GOLD_DATASET_NAME[GOLD_DATASET_NAME != ""]

if len(GOLD_DATASET_NAME) == 0:
    raise ValueError("Gold cascade input dataframe is missing usable meta__dataset values.")

GOLD_DATASET_NAME = str(GOLD_DATASET_NAME.iloc[0]).strip()

gold_truth = load_parent_truth_record_from_dataframe(
    dataframe=gold_preprocessed_scaled_dataframe,
    truth_dir=TRUTHS_PATH,
    parent_layer_name="gold",
    dataset_name=GOLD_DATASET_NAME,
    column_name="meta__truth_hash",
)

DATASET_NAME = get_dataset_name_from_truth(gold_truth)
GOLD_PARENT_TRUTH_HASH = get_truth_hash(gold_truth)

PARENT_PIPELINE_MODE = get_pipeline_mode_from_truth(gold_truth)
if PARENT_PIPELINE_MODE is not None:
    PIPELINE_MODE = PARENT_PIPELINE_MODE

GOLD_TRUTH_PATH = (
    TRUTHS_PATH
    / "gold"
    / f"{DATASET_NAME}__gold__truth__{GOLD_PARENT_TRUTH_HASH}.json"
)

gold_truth_runtime_facts = gold_truth.get("runtime_facts", {})
gold_truth_artifact_paths = gold_truth.get("artifact_paths", {})

GOLD_PREPROCESSED_DATA_PATH = Path(gold_truth_artifact_paths.get("gold_preprocessed_path", str(GOLD_PREPROCESSED_DATA_PATH)))
GOLD_FIT_DATA_PATH = Path(gold_truth_artifact_paths.get("gold_fit_path", str(GOLD_FIT_DATA_PATH)))
GOLD_TEST_DATA_PATH = Path(gold_truth_artifact_paths.get("gold_test_path", str(GOLD_TEST_DATA_PATH)))
GOLD_TRAIN_DATA_PATH = Path(gold_truth_artifact_paths.get("gold_train_path", str(GOLD_TRAIN_DATA_PATH)))
STAGE1_FEATURES_PATH = Path(gold_truth_artifact_paths.get("stage1_features_path", str(STAGE1_FEATURES_PATH)))
STAGE2_FEATURES_PATH = Path(gold_truth_artifact_paths.get("stage2_features_path", str(STAGE2_FEATURES_PATH)))
STAGE3_PRIMARY_PATH = Path(gold_truth_artifact_paths.get("stage3_primary_path", str(STAGE3_PRIMARY_PATH)))
STAGE3_SECONDARY_PATH = Path(gold_truth_artifact_paths.get("stage3_secondary_path", str(STAGE3_SECONDARY_PATH)))

logger.info("Resolved Gold cascade dataset name from Gold truth: %s", DATASET_NAME)
logger.info("Resolved Gold truth path: %s", GOLD_TRUTH_PATH)

print("Gold cascade dataset name from parent truth:", DATASET_NAME)
print("Gold cascade parent truth hash:", GOLD_PARENT_TRUTH_HASH)

logger.info("Loading Gold Preprocessed parquet: %s", GOLD_PREPROCESSED_DATA_PATH)
gold_preprocessed_dataframe = load_data(GOLD_PREPROCESSED_DATA_PATH)

logger.info("Loading Gold fit parquet: %s", GOLD_FIT_DATA_PATH)
gold_fit_dataframe = load_data(GOLD_FIT_DATA_PATH)

logger.info("Loading Gold test parquet: %s", GOLD_TEST_DATA_PATH)
gold_test_dataframe = load_data(GOLD_TEST_DATA_PATH)

logger.info("Loading Gold train parquet: %s", GOLD_TRAIN_DATA_PATH)
gold_train_dataframe = load_data(GOLD_TRAIN_DATA_PATH)

logger.info("Loading Stage 1 features: %s", STAGE1_FEATURES_PATH)
stage1_feature_columns = load_json(STAGE1_FEATURES_PATH)

logger.info("Loading Stage 2 features: %s", STAGE2_FEATURES_PATH)
stage2_feature_columns = load_json(STAGE2_FEATURES_PATH)

logger.info("Loading Stage 3 primary rule sensors: %s", STAGE3_PRIMARY_PATH)
stage3_primary_rule_sensors = load_json(STAGE3_PRIMARY_PATH)

logger.info("Loading Stage 3 secondary rule sensors: %s", STAGE3_SECONDARY_PATH)
stage3_secondary_rule_sensors = load_json(STAGE3_SECONDARY_PATH)

logger.info("Loading Tuned Stage 2 Parameters: %s", CASCADE_TUNED_THRESHOLDS_PATH)
cascade_tuned_thresholds  = load_json(CASCADE_TUNED_THRESHOLDS_PATH)

ledger.add(
    kind="step",
    step="load_modeling_inputs",
    message="Loaded Gold scaled parquet, loaded Gold truth, substituted truth-linked artifact paths, then loaded cascade inputs.",
    data={
        "gold_scaled_path": str(GOLD_PREPROCESSED_SCALED_DATA_PATH),
        "gold_truth_hash": GOLD_PARENT_TRUTH_HASH,
        "gold_truth_path": str(GOLD_TRUTH_PATH),
        "gold_preprocessed_path": str(GOLD_PREPROCESSED_DATA_PATH),
        "gold_fit_path": str(GOLD_FIT_DATA_PATH),
        "gold_test_path": str(GOLD_TEST_DATA_PATH),
        "gold_train_path": str(GOLD_TRAIN_DATA_PATH),
        "stage1_features_path": str(STAGE1_FEATURES_PATH),
        "stage2_features_path": str(STAGE2_FEATURES_PATH),
        "stage3_primary_path": str(STAGE3_PRIMARY_PATH),
        "stage3_secondary_path": str(STAGE3_SECONDARY_PATH),
        "gold_preprocessed_shape": list(gold_preprocessed_dataframe.shape),
        "gold_scaled_shape": list(gold_preprocessed_scaled_dataframe.shape),
        "gold_fit_shape": list(gold_fit_dataframe.shape),
        "gold_test_shape": list(gold_test_dataframe.shape),
        "gold_train_shape": list(gold_train_dataframe.shape),
        "stage1_feature_count": int(len(stage1_feature_columns)),
        "stage2_feature_count": int(len(stage2_feature_columns)),
        "stage3_primary_count": int(len(stage3_primary_rule_sensors)),
        "stage3_secondary_count": int(len(stage3_secondary_rule_sensors)),
    },
    logger=logger,
)

gold_test_dataframe.head(3)

2026-03-23 23:43:39,882 | INFO | capstone.gold | Loading Gold Preprocessed parquet: /workspace/data/gold/pump__gold__preprocessed_scaled.parquet
2026-03-23 23:43:39,886 | INFO | capstone.file_io | Loading Parquet: /workspace/data/gold/pump__gold__preprocessed_scaled.parquet


2026-03-23 23:43:42,536 | INFO | capstone.gold | Resolved Gold cascade dataset name from Gold truth: pump
2026-03-23 23:43:42,538 | INFO | capstone.gold | Resolved Gold truth path: /workspace/artifacts/truths/gold/pump__gold__truth__8a62f44686a717446607a3d0303285188fa60d19339eef92a626f729f6333117.json
2026-03-23 23:43:42,541 | INFO | capstone.gold | Loading Gold Preprocessed parquet: /workspace/data/gold/pump__gold__preprocessed_prescaled.parquet
2026-03-23 23:43:42,543 | INFO | capstone.file_io | Loading Parquet: /workspace/data/gold/pump__gold__preprocessed_prescaled.parquet


Gold cascade dataset name from parent truth: pump
Gold cascade parent truth hash: 8a62f44686a717446607a3d0303285188fa60d19339eef92a626f729f6333117


2026-03-23 23:43:46,633 | INFO | capstone.gold | Loading Gold fit parquet: /workspace/data/gold/pump__gold__fit_normal_only.parquet
2026-03-23 23:43:46,637 | INFO | capstone.file_io | Loading Parquet: /workspace/data/gold/pump__gold__fit_normal_only.parquet
2026-03-23 23:43:47,647 | INFO | capstone.gold | Loading Gold test parquet: /workspace/data/gold/pump__gold__test.parquet
2026-03-23 23:43:47,650 | INFO | capstone.file_io | Loading Parquet: /workspace/data/gold/pump__gold__test.parquet
2026-03-23 23:43:48,025 | INFO | capstone.gold | Loading Gold train parquet: /workspace/data/gold/pump__gold__train.parquet
2026-03-23 23:43:48,027 | INFO | capstone.file_io | Loading Parquet: /workspace/data/gold/pump__gold__train.parquet
2026-03-23 23:43:49,287 | INFO | capstone.gold | Loading Stage 1 features: /workspace/artifacts/gold/pump/pump__gold__stage1_features.json
2026-03-23 23:43:49,297 | INFO | capstone.file_io | Loading JSON: /workspace/artifacts/gold/pump/pump__gold__stage1_features.j

,meta__asset_id,meta__dataset,meta__episode_id,meta__event_id,meta__ingested_at_utc,meta__parent_truth_hash,meta__pipeline_mode,meta__record_id,meta__run_id,meta__source_file,meta__source_row_id,meta__split,meta__truth_hash,event_time,event_step,time_index,event_date,anomaly_flag,is_anomaly,is_normal,status_normal_value,sensor_00,sensor_01,sensor_02,sensor_03,sensor_04,sensor_05,sensor_06,sensor_07,sensor_08,sensor_09,sensor_10,sensor_11,sensor_12,sensor_13,sensor_14,sensor_16,sensor_17,sensor_18,sensor_19,sensor_20,sensor_21,sensor_22,sensor_23,sensor_24,sensor_25,sensor_26,sensor_27,sensor_28,sensor_29,sensor_30,sensor_31,sensor_32,sensor_33,sensor_34,sensor_35,sensor_36,sensor_37,sensor_38,sensor_39,sensor_40,sensor_41,sensor_42,sensor_43,sensor_44,sensor_45,sensor_46,sensor_47,sensor_48,sensor_49,sensor_51,timestamp,machine_status,meta__is_train_flag
0,asset__001,pump,5,pump:asset__001:run__001:136431,2026-03-22 00:55:01.594555+00:00,150ec91684fac04d2ee1c41c1b9297bde4fcd92fa30a2f...,batch,9701747606095593835,run__001,sensor.csv,136431,test,8a62f44686a717446607a3d0303285188fa60d19339eef...,2018-07-04 17:51:00+00:00,136431,136431,2018-07-04 00:00:00+00:00,0,0,1,NORMAL,-50.814252,-2.274511,-5.888888,-1.236360,-50.126917,-9.021832,-6.255302,-6.828505,-6.049950,-11.333579,-6.540518,-5.119545,-5.997311,-0.902079,0.043854,0.025437,0.189619,0.158404,-0.098290,0.006694,0.050789,0.674963,3.116164,0.436222,0.237276,0.492741,0.629638,0.191590,-0.772389,0.780644,0.234333,0.703432,-0.216653,0.677263,1.453255,0.406195,0.319102,-1.437501,8.590904,-0.109091,-0.863635,20.823540,6.666667,-1.181816,-1.136364,-1.035715,-1.26087,-0.941635,-2.166668,-4.21168,2018-07-04 17:51:00,NORMAL,False
1,asset__001,pump,5,pump:asset__001:run__001:136432,2026-03-22 00:55:01.594555+00:00,150ec91684fac04d2ee1c41c1b9297bde4fcd92fa30a2f...,batch,4242984989007806471,run__001,sensor.csv,136432,test,8a62f44686a717446607a3d0303285188fa60d19339eef...,2018-07-04 17:52:00+00:00,136432,136432,2018-07-04 00:00:00+00:00,0,0,1,NORMAL,-50.814253,-2.098039,-5.888888,-1.272725,-49.945099,-9.079268,-6.361687,-6.928511,-6.049950,-11.333579,-6.566318,-5.119545,-5.997311,-0.902079,0.089622,-0.077443,0.357106,0.361669,-0.017151,0.009255,0.056949,0.698316,3.095068,0.416996,0.250370,0.493630,0.425490,0.269709,-0.708270,1.090321,0.975476,0.786631,-0.172121,0.590545,1.320089,0.404733,-0.098002,-1.468751,8.363630,0.000000,-0.909090,21.058842,7.625000,-1.181816,-1.227272,-1.035714,-1.26087,-0.941635,-2.166668,-4.21168,2018-07-04 17:52:00,NORMAL,False
2,asset__001,pump,5,pump:asset__001:run__001:136433,2026-03-22 00:55:01.594555+00:00,150ec91684fac04d2ee1c41c1b9297bde4fcd92fa30a2f...,batch,11874558860056305371,run__001,sensor.csv,136433,test,8a62f44686a717446607a3d0303285188fa60d19339eef...,2018-07-04 17:53:00+00:00,136433,136433,2018-07-04 00:00:00+00:00,0,0,1,NORMAL,-50.814253,-1.941177,-5.888888,-1.290906,-50.013281,-9.088074,-6.361687,-6.871359,-6.033293,-11.500207,-6.563817,-5.119545,-5.997311,-0.902079,0.013262,-0.098342,-0.066365,-0.005373,-0.042199,0.069460,0.026548,0.665235,3.039044,0.376989,0.231377,0.507400,0.502814,0.268206,-0.661707,0.677419,0.858314,0.791715,0.050019,0.509348,1.203625,0.393889,0.376575,-1.468751,7.954540,0.072728,-0.909090,20.705878,8.375000,-1.181816,-1.227273,-1.035714,-1.26087,-0.941635,-2.166668,-4.21168,2018-07-04 17:53:00,NORMAL,False


----

## Rebuild the Train and Test Masks

Before fitting or evaluating the cascade, I recover the train/test split that was already stamped during Gold preprocessing.

The key point here is that this notebook is **not** creating a new split. It is reusing the split created upstream so the baseline notebook, the default cascade notebook, the tuned cascade notebook, and the comparison notebook all work from the same row partition.

In [63]:
# Masks
if "meta__is_train_flag" not in gold_preprocessed_scaled_dataframe.columns:
    raise ValueError("meta__is_train_flag missing from gold_preprocessed_scaled_dataframe. "
                     "Gold preprocessing must stamp it before saving.")

train_mask = gold_preprocessed_scaled_dataframe["meta__is_train_flag"].astype(bool)
test_mask = ~train_mask

logger.info("Split counts: all=%d train=%d test=%d", len(train_mask), int(train_mask.sum()), int(test_mask.sum()))

2026-03-23 23:43:49,874 | INFO | capstone.gold | Split counts: all=220320 train=136431 test=83889


### Ask

Why does the tuned cascade notebook reuse the saved split instead of building a fresh one here?

### Answer

I want the evaluation setup to stay consistent across the Gold modeling notebooks.

If the baseline and cascade notebooks each made their own split, then the results would be harder to compare fairly. By reusing the saved Gold split, I know each modeling approach is being judged against the same training and test boundary.

So this step is mainly about consistency and comparability.

----

## Define the Stage 3 Reference Profile Logic

Before Stage 3 can confirm alerts, I need a reference profile that describes what normal feature behavior looks like.

This helper function builds that profile from the Gold fit dataframe by summarizing each selected feature using values like:
- median
- mean
- standard deviation
- lower bound
- upper bound

The main idea is that Stage 3 should not just look for model flags. It should also compare suspicious rows against a stored picture of normal behavior.

In [64]:
def build_reference_profile(
    dataframe: pd.DataFrame,
    *,
    feature_columns: list[str],
) -> pd.DataFrame:
    reference_rows: list[dict] = []

    for feature_name in feature_columns:
        feature_series = dataframe[feature_name]

        reference_rows.append({
            "feature_name": feature_name,
            "median_value": float(feature_series.median()),
            "mean_value": float(feature_series.mean()),
            "standard_deviation": float(feature_series.std()) if not pd.isna(feature_series.std()) else 0.0,
            "lower_bound": float(feature_series.quantile(0.05)),
            "upper_bound": float(feature_series.quantile(0.95)),
        })

    reference_profile = pd.DataFrame(reference_rows)
    return reference_profile

## Build the Stage 3 Reference Profile

Here I create the actual reference profile used by Stage 3 confirmation logic.

The feature set for this profile combines:
- the broader Stage 1 modeling features
- the Stage 3 primary rule sensors
- the Stage 3 secondary rule sensors

That gives Stage 3 a normal-behavior reference it can use when checking for profile breaches and rule evidence.

In [65]:
reference_profile_features = list(dict.fromkeys(
    stage1_feature_columns + stage3_primary_rule_sensors + stage3_secondary_rule_sensors
))

reference_profile = build_reference_profile(
    gold_fit_dataframe,
    feature_columns=reference_profile_features,
)

ledger.add(
    kind="step",
    step="build_reference_profile",
    message="Built fit-period reference profile for Stage 3 confirmation logic.",
    data={
        "training_rows": int(len(gold_fit_dataframe)),
        "reference_feature_count": int(len(reference_profile_features)),
    },
    logger=logger,
)

reference_profile.head(10)

2026-03-23 23:43:50,867 | INFO | capstone.gold | LEDGER | {'ts_utc': '2026-03-23T23:43:50.867332+00:00', 'stage': 'gold', 'recipe': 'gold_modeling__v001_cascade', 'kind': 'step', 'step': 'build_reference_profile', 'message': 'Built fit-period reference profile for Stage 3 confirmation logic.', 'why': None, 'consequence': None, 'data': {'training_rows': 122065, 'reference_feature_count': 50}}


,feature_name,median_value,mean_value,standard_deviation,lower_bound,upper_bound
0,sensor_00,0.0,-1.191161,6.117528,-6.418646,1.302319
1,sensor_01,0.0,0.014146,0.838560,-1.313726,1.431370
2,sensor_02,0.0,-0.045551,0.814930,-1.259258,1.055557
3,sensor_03,0.0,-0.003276,0.729044,-1.145452,1.145452
4,sensor_04,0.0,-0.874445,5.304649,-3.770447,1.181813
5,sensor_05,0.0,-0.036448,1.103071,-1.376761,1.517739
6,sensor_06,0.0,-0.212238,1.983681,-1.425542,1.808494
7,sensor_07,0.0,-0.275468,1.148624,-1.085688,1.028576
8,sensor_08,0.0,0.042985,1.223742,-0.949980,1.283344
9,sensor_09,0.0,-0.434937,2.478143,-1.700060,0.366665


----

## Prepare the Feature Matrices and Evaluation Labels

Now I build the actual feature matrices that the cascade will use.

This includes:
- Stage 1 fit features from the Gold fit dataframe
- Stage 2 fit features from the Gold fit dataframe
- Stage 1 full-dataset scoring features
- Stage 2 full-dataset scoring features
- test labels, when `anomaly_flag` is available

A detail that matters here is that the model fitting happens on the **normal-only Gold fit subset**, while scoring happens across the **full scaled Gold dataset**.

In [66]:
# Fit features from normal-only fit parquet
stage1_train_fit_features = gold_fit_dataframe[stage1_feature_columns].values
stage2_train_fit_features = gold_fit_dataframe[stage2_feature_columns].values

# Score features from the full scaled dataset (ALL rows)
stage1_all_features = gold_preprocessed_scaled_dataframe[stage1_feature_columns].values
stage2_all_features = gold_preprocessed_scaled_dataframe[stage2_feature_columns].values


test_labels = None

if "anomaly_flag" in gold_preprocessed_scaled_dataframe.columns:
    test_labels = (
        gold_preprocessed_scaled_dataframe
        .loc[test_mask, "anomaly_flag"]
        .fillna(0)
        .astype(int)
        .values
    )

----

### Ask

Why am I fitting on the Gold fit subset but scoring the full scaled dataset?

### Answer

Because those steps are doing different jobs.

The fit subset gives the cascade its reference view of normal behavior. The full scaled dataset is what I want to score afterward so I can see where alerts appear across all rows.

So the overall logic is:
- fit the Stage 1 and Stage 2 models on the saved normal training subset
- score every row in the scaled Gold dataset
- evaluate the final cascade outcome on the held-out test rows

In [67]:
def compute_anomaly_scores_isolation_forest(
    model: IsolationForest,
    feature_matrix: np.ndarray,
) -> np.ndarray:
    scores = model.score_samples(feature_matrix)
    anomaly_scores = -scores
    return anomaly_scores

In [68]:
def choose_threshold_by_percentile(
    anomaly_scores: np.ndarray,
    percentile: float,
) -> float:
    return float(np.percentile(anomaly_scores, percentile))

In [69]:

def evaluate_against_labels(
    true_labels: np.ndarray,
    anomaly_scores: np.ndarray,
    threshold: float,
) -> dict:
    predicted_labels = (anomaly_scores >= threshold).astype(int)

    precision, recall, f1, _ = precision_recall_fscore_support(
        true_labels,
        predicted_labels,
        average="binary",
        zero_division=0,
    )

    results = {
        "threshold": float(threshold),
        "precision": float(precision),
        "recall": float(recall),
        "f1": float(f1),
    }

    if len(np.unique(true_labels)) == 2:
        results["roc_auc"] = float(roc_auc_score(true_labels, anomaly_scores))
        results["pr_auc"] = float(average_precision_score(true_labels, anomaly_scores))
    else:
        results["roc_auc"] = None
        results["pr_auc"] = None

    return results

----

## Run Stage 1: Broad Isolation Forest Screening

This is the first model stage of the cascade.

Stage 1 uses the broader feature set to act as an initial screen. The goal here is to cast a wider net and identify rows that look suspicious enough to move forward in the cascade.

In this step I:
- fit the Stage 1 Isolation Forest on the Gold fit subset
- score the fit subset and the full scaled dataset
- choose the Stage 1 threshold from the training-score distribution
- create the Stage 1 alert flags

This stage is intentionally broad. It is supposed to surface candidate alerts, not be the final decision by itself.

In [70]:
stage1_model = IsolationForest(
    n_estimators=STAGE1_ESTIMATOR_COUNT,
    random_state=RANDOM_SEED,
    n_jobs=-1,
)

stage1_model.fit(stage1_train_fit_features)

stage1_train_scores = compute_anomaly_scores_isolation_forest(
    stage1_model, 
    stage1_train_fit_features
)

stage1_all_scores = compute_anomaly_scores_isolation_forest(
    stage1_model, 
    stage1_all_features
)

stage1_threshold = choose_threshold_by_percentile(
    stage1_train_scores, 
    STAGE1_THRESHOLD_PERCENTILE
)

stage1_flags = (stage1_all_scores >= stage1_threshold).astype(int)

stage1_summary = {
    "threshold_percentile": float(STAGE1_THRESHOLD_PERCENTILE),
    "threshold": float(stage1_threshold),
    "alert_count_all_rows": int(stage1_flags.sum()),
    "alert_count_test_rows": int(stage1_flags[test_mask.values].sum()),
}

ledger.add(
    kind="step",
    step="run_cascade_stage1",
    message="Ran Stage 1 broad Isolation Forest using saved Gold fit data and scored all rows of the scaled dataset.",
    data={
        "estimator_count": int(STAGE1_ESTIMATOR_COUNT),
        "threshold_percentile": float(STAGE1_THRESHOLD_PERCENTILE),
        "threshold": float(stage1_threshold),
        "feature_count": int(len(stage1_feature_columns)),
        "alert_count_all_rows": int(stage1_summary["alert_count_all_rows"]),
        "alert_count_test_rows": int(stage1_summary["alert_count_test_rows"]),
    },
    logger=logger,
)

2026-03-23 23:43:56,187 | INFO | capstone.gold | LEDGER | {'ts_utc': '2026-03-23T23:43:56.187360+00:00', 'stage': 'gold', 'recipe': 'gold_modeling__v001_cascade', 'kind': 'step', 'step': 'run_cascade_stage1', 'message': 'Ran Stage 1 broad Isolation Forest using saved Gold fit data and scored all rows of the scaled dataset.', 'why': None, 'consequence': None, 'data': {'estimator_count': 200, 'threshold_percentile': 95.0, 'threshold': 0.5337445171501773, 'feature_count': 50, 'alert_count_all_rows': 22742, 'alert_count_test_rows': 2288}}


{'ts_utc': '2026-03-23T23:43:56.187360+00:00',
 'stage': 'gold',
 'recipe': 'gold_modeling__v001_cascade',
 'kind': 'step',
 'step': 'run_cascade_stage1',
 'message': 'Ran Stage 1 broad Isolation Forest using saved Gold fit data and scored all rows of the scaled dataset.',
 'why': None,
 'consequence': None,
 'data': {'estimator_count': 200,
  'threshold_percentile': 95.0,
  'threshold': 0.5337445171501773,
  'feature_count': 50,
  'alert_count_all_rows': 22742,
  'alert_count_test_rows': 2288}}

### Ask

What is Stage 1 really supposed to do in the cascade?

### Answer

Stage 1 is the broad screening step.

Its job is to catch rows that look suspicious enough to deserve a second look. I do not expect Stage 1 alone to be highly selective. In fact, it is normal for Stage 1 to raise more alerts than the later stages, because the later stages are there to narrow and confirm those alerts.

So when I review Stage 1, I mainly care that it is acting as a reasonable first filter rather than as the final answer.

----

## Define the Stage 2 Selection Logic

This helper section defines how the tuned Stage 2 model is chosen.

Instead of assuming one fixed Stage 2 setup, this notebook evaluates candidate model and threshold combinations and keeps the best result based on the configured selection score. That score is designed to reward stronger alert quality while still respecting the minimum recall requirement.

So this is the part of the notebook where the tuned cascade starts to differ from the default cascade version.

In [71]:
def evaluate_stage2_model_with_thresholds(
    *,
    model: IsolationForest,
    model_params: dict,
    stage2_train_fit_features: np.ndarray,
    stage2_all_features: np.ndarray,
    stage1_flags: np.ndarray,
    test_mask: pd.Series,
    test_labels: np.ndarray | None,
    threshold_percentiles: list[float],
    min_recall: float,
) -> dict:
    model.fit(stage2_train_fit_features)

    stage2_train_scores = compute_anomaly_scores_isolation_forest(
        model,
        stage2_train_fit_features,
    )

    stage2_all_scores = compute_anomaly_scores_isolation_forest(
        model,
        stage2_all_features,
    )

    best_result = None

    for threshold_percentile in threshold_percentiles:
        stage2_threshold = choose_threshold_by_percentile(
            stage2_train_scores,
            float(threshold_percentile),
        )

        stage2_raw_flags = (stage2_all_scores >= stage2_threshold).astype(int)
        stage2_flags = ((stage1_flags == 1) & (stage2_raw_flags == 1)).astype(int)

        if test_labels is not None:
            stage2_test_flags = stage2_flags[test_mask.values]

            precision, recall, f1, _ = precision_recall_fscore_support(
                test_labels,
                stage2_test_flags,
                average="binary",
                zero_division=0,
            )

            tn, fp, fn, tp = confusion_matrix(
                test_labels,
                stage2_test_flags,
                labels=[0, 1],
            ).ravel()

            if recall < min_recall:
                selection_score = -1.0 + float(recall)
            else:
                selection_score = (
                    (float(precision) * 0.50)
                    + (float(recall) * 0.20)
                    + (float(f1) * 0.30)
                )
        else:
            precision = None
            recall = None
            f1 = None
            tn = None
            fp = None
            fn = None
            tp = None
            selection_score = -float(stage2_flags[test_mask.values].sum())

        candidate_result = {
            "model_params": model_params,
            "selected_threshold_percentile": float(threshold_percentile),
            "stage2_threshold": float(stage2_threshold),
            "stage2_train_scores": stage2_train_scores,
            "stage2_all_scores": stage2_all_scores,
            "stage2_raw_flags": stage2_raw_flags,
            "stage2_flags": stage2_flags,
            "raw_alert_count_all_rows": int(stage2_raw_flags.sum()),
            "raw_alert_count_test_rows": int(stage2_raw_flags[test_mask.values].sum()),
            "stage2_confirmed_count_all_rows": int(stage2_flags.sum()),
            "stage2_confirmed_count_test_rows": int(stage2_flags[test_mask.values].sum()),
            "precision": None if precision is None else float(precision),
            "recall": None if recall is None else float(recall),
            "f1": None if f1 is None else float(f1),
            "selection_score": float(selection_score),
            "tn": None if tn is None else int(tn),
            "fp": None if fp is None else int(fp),
            "fn": None if fn is None else int(fn),
            "tp": None if tp is None else int(tp),
        }

        if (
            best_result is None
            or candidate_result["selection_score"] > best_result["selection_score"]
        ):
            best_result = candidate_result

    if best_result is None:
        raise ValueError("Stage 2 threshold evaluation did not produce a valid result.")

    return best_result


def run_stage2_selection(
    *,
    selection_mode: str,
    fixed_params: dict,
    fixed_threshold_percentile: float,
    threshold_grid: list[float],
    param_grid: dict,
    stage2_train_fit_features: np.ndarray,
    stage2_all_features: np.ndarray,
    stage1_flags: np.ndarray,
    test_mask: pd.Series,
    test_labels: np.ndarray | None,
    random_seed: int,
    min_recall: float,
) -> tuple[IsolationForest, dict, pd.DataFrame]:
    selection_mode_clean = str(selection_mode).strip().lower()

    search_rows: list[dict] = []
    best_result = None
    best_model = None

    if selection_mode_clean == "fixed":
        model_candidates = [fixed_params.copy()]
        threshold_candidates = [float(fixed_threshold_percentile)]

    elif selection_mode_clean == "threshold_grid":
        model_candidates = [fixed_params.copy()]
        threshold_candidates = [float(value) for value in threshold_grid]

    elif selection_mode_clean == "parameter_search":
        model_candidates = [dict(params) for params in ParameterGrid(param_grid)]
        threshold_candidates = [float(value) for value in threshold_grid]

    else:
        raise ValueError(
            f"Unsupported STAGE2_SELECTION_MODE: {selection_mode}. "
            "Use 'fixed', 'threshold_grid', or 'parameter_search'."
        )

    for model_params in model_candidates:
        candidate_model = IsolationForest(
            random_state=random_seed,
            n_jobs=-1,
            **model_params,
        )

        candidate_result = evaluate_stage2_model_with_thresholds(
            model=candidate_model,
            model_params=model_params,
            stage2_train_fit_features=stage2_train_fit_features,
            stage2_all_features=stage2_all_features,
            stage1_flags=stage1_flags,
            test_mask=test_mask,
            test_labels=test_labels,
            threshold_percentiles=threshold_candidates,
            min_recall=min_recall,
        )

        search_rows.append({
            **candidate_result["model_params"],
            "selected_threshold_percentile": candidate_result["selected_threshold_percentile"],
            "stage2_threshold": candidate_result["stage2_threshold"],
            "raw_alert_count_all_rows": candidate_result["raw_alert_count_all_rows"],
            "raw_alert_count_test_rows": candidate_result["raw_alert_count_test_rows"],
            "stage2_confirmed_count_all_rows": candidate_result["stage2_confirmed_count_all_rows"],
            "stage2_confirmed_count_test_rows": candidate_result["stage2_confirmed_count_test_rows"],
            "precision": candidate_result["precision"],
            "recall": candidate_result["recall"],
            "f1": candidate_result["f1"],
            "selection_score": candidate_result["selection_score"],
            "tn": candidate_result["tn"],
            "fp": candidate_result["fp"],
            "fn": candidate_result["fn"],
            "tp": candidate_result["tp"],
        })

        if (
            best_result is None
            or candidate_result["selection_score"] > best_result["selection_score"]
        ):
            best_result = candidate_result
            best_model = candidate_model

    if best_result is None or best_model is None:
        raise ValueError("Stage 2 selection did not produce a best model.")

    search_results = pd.DataFrame(search_rows).sort_values(
        by=["selection_score"],
        ascending=[False],
    ).reset_index(drop=True)

    return best_model, best_result, search_results

### Ask

What is the actual tuning idea behind Stage 2 here?

### Answer

The point is not to blindly search for the biggest score. The point is to choose a narrower Stage 2 setup that still behaves well on the held-out evaluation logic.

In this notebook, each candidate setup is judged using:
- the selected threshold percentile
- the resulting confirmed alert counts
- precision, recall, and F1 on the test rows when labels are available
- a selection score that gives the most weight to precision, while still keeping recall and F1 involved
- a minimum recall rule so overly strict candidates do not automatically win

So Stage 2 tuning here is really a controlled search for a better confirmation layer, not a generic hyperparameter hunt.

----

In [72]:
cascade_tuned_thresholds = load_json(CASCADE_TUNED_THRESHOLDS_PATH)

STAGE2_SELECTION_MODE = "fixed"

STAGE2_FIXED_PARAMS = dict(cascade_tuned_thresholds["stage2_best_params"])
STAGE2_FIXED_THRESHOLD_PERCENTILE = float(
    cascade_tuned_thresholds["stage2_selected_threshold_percentile"]
)

2026-03-23 23:43:56,852 | INFO | capstone.file_io | Loading JSON: /workspace/artifacts/gold/pump/pump__gold__cascade_tuned_thresholds.json


----

## Run Stage 2 Selection and Keep the Best Narrow Model

This is the tuned Stage 2 step of the cascade.

Here I run the configured Stage 2 selection process, which may operate in one of several modes:
- fixed
- threshold grid
- parameter search

For the tuned cascade, the goal is to search the candidate space and keep the narrow Isolation Forest setup that gives the best selection result under the notebook's rules.

After that, I keep:
- the best Stage 2 model
- the selected threshold percentile
- the chosen Stage 2 threshold
- the best parameter set
- the full search results table for review

In [73]:
stage2_model, best_stage2_result, stage2_search_results = run_stage2_selection(
    selection_mode=STAGE2_SELECTION_MODE,
    fixed_params=STAGE2_FIXED_PARAMS,
    fixed_threshold_percentile=STAGE2_FIXED_THRESHOLD_PERCENTILE,
    threshold_grid=STAGE2_THRESHOLD_GRID,
    param_grid=STAGE2_PARAM_GRID,
    stage2_train_fit_features=stage2_train_fit_features,
    stage2_all_features=stage2_all_features,
    stage1_flags=stage1_flags,
    test_mask=test_mask,
    test_labels=test_labels,
    random_seed=STAGE2_RANDOM_STATE,
    min_recall=STAGE2_MIN_RECALL,
)

stage2_train_scores = best_stage2_result["stage2_train_scores"]
stage2_all_scores = best_stage2_result["stage2_all_scores"]
stage2_threshold = best_stage2_result["stage2_threshold"]
stage2_raw_flags = best_stage2_result["stage2_raw_flags"]
stage2_flags = best_stage2_result["stage2_flags"]

stage2_selected_threshold_percentile = best_stage2_result["selected_threshold_percentile"]
stage2_best_params = best_stage2_result["model_params"]

stage2_summary = {
    "selection_mode": STAGE2_SELECTION_MODE,
    "selected_threshold_percentile": float(stage2_selected_threshold_percentile),
    "threshold": float(stage2_threshold),
    "best_params": stage2_best_params,
    "raw_alert_count_all_rows": int(best_stage2_result["raw_alert_count_all_rows"]),
    "raw_alert_count_test_rows": int(best_stage2_result["raw_alert_count_test_rows"]),
    "stage2_confirmed_count_all_rows": int(best_stage2_result["stage2_confirmed_count_all_rows"]),
    "stage2_confirmed_count_test_rows": int(best_stage2_result["stage2_confirmed_count_test_rows"]),
    "precision": float(best_stage2_result["precision"]),
    "recall": float(best_stage2_result["recall"]),
    "f1": float(best_stage2_result["f1"]),
    "selection_score": float(best_stage2_result["selection_score"]),
    "candidate_count": int(len(stage2_search_results)),
}

ledger.add(
    kind="step",
    step="run_cascade_stage2_selection",
    message="Ran Stage 2 selection using the configured mode and chose the best narrow Isolation Forest configuration.",
    data=stage2_summary,
    logger=logger,
)

stage2_search_results.head(10)

2026-03-23 23:43:58,644 | INFO | capstone.gold | LEDGER | {'ts_utc': '2026-03-23T23:43:58.644555+00:00', 'stage': 'gold', 'recipe': 'gold_modeling__v001_cascade', 'kind': 'step', 'step': 'run_cascade_stage2_selection', 'message': 'Ran Stage 2 selection using the configured mode and chose the best narrow Isolation Forest configuration.', 'why': None, 'consequence': None, 'data': {'selection_mode': 'fixed', 'selected_threshold_percentile': 99.0, 'threshold': 0.6078905948669239, 'best_params': {'bootstrap': False, 'contamination': 'auto', 'max_features': 1.0, 'max_samples': 'auto', 'n_estimators': 100}, 'raw_alert_count_all_rows': 14612, 'raw_alert_count_test_rows': 1309, 'stage2_confirmed_count_all_rows': 13898, 'stage2_confirmed_count_test_rows': 648, 'precision': 0.09259259259259259, 'recall': 0.5084745762711864, 'f1': 0.1566579634464752, 'selection_score': 0.19498860058447615, 'candidate_count': 1}}


,bootstrap,contamination,max_features,max_samples,n_estimators,selected_threshold_percentile,stage2_threshold,raw_alert_count_all_rows,raw_alert_count_test_rows,stage2_confirmed_count_all_rows,stage2_confirmed_count_test_rows,precision,recall,f1,selection_score,tn,fp,fn,tp
0,False,auto,1.0,auto,100,99.0,0.607891,14612,1309,13898,648,0.092593,0.508475,0.156658,0.194989,83183,588,58,60


### Ask

What should I pay attention to when I look at the Stage 2 search results?

### Answer

The main things I care about are:
- what `selection_mode` was used
- how many candidate combinations were evaluated
- which parameter set won
- which threshold percentile was selected
- how many raw Stage 2 alerts were produced
- how many confirmed Stage 2 alerts survived after the Stage 1 gate
- whether the winning candidate looks balanced rather than simply aggressive or overly quiet

This is important because the tuned cascade should not just be "different" from the default one. It should have a clear reason for why its Stage 2 branch was selected.

----

#### Quick Verifications Cell 

In [74]:
print("STAGE2_SELECTION_MODE:", STAGE2_SELECTION_MODE)
print("stage2_best_params:", stage2_best_params)
print("search candidate count:", len(stage2_search_results) if "stage2_search_results" in globals() else "missing")

if "stage2_search_results" in globals() and len(stage2_search_results) > 0:
    display(pd.DataFrame(stage2_search_results).head())

STAGE2_SELECTION_MODE: fixed
stage2_best_params: {'bootstrap': False, 'contamination': 'auto', 'max_features': 1.0, 'max_samples': 'auto', 'n_estimators': 100}
search candidate count: 1


,bootstrap,contamination,max_features,max_samples,n_estimators,selected_threshold_percentile,stage2_threshold,raw_alert_count_all_rows,raw_alert_count_test_rows,stage2_confirmed_count_all_rows,stage2_confirmed_count_test_rows,precision,recall,f1,selection_score,tn,fp,fn,tp
0,False,auto,1.0,auto,100,99.0,0.607891,14612,1309,13898,648,0.092593,0.508475,0.156658,0.194989,83183,588,58,60


----

## Build the Main Cascade Results Dataframe

Now I create the working results dataframe that will hold the outputs from each cascade stage.

At this point I attach the row-level Stage 1 and Stage 2 outputs, including:
- `stage1_score`
- `stage1_flag`
- `stage2_score`
- `stage2_raw_flag`
- `stage2_flag`

This creates a single dataframe that the Stage 3 rule logic can build on next.

In [75]:
cascade_results = gold_preprocessed_scaled_dataframe.copy()

cascade_results["stage1_score"] = stage1_all_scores
cascade_results["stage1_flag"] = stage1_flags

cascade_results["stage2_score"] = stage2_all_scores
cascade_results["stage2_raw_flag"] = stage2_raw_flags
cascade_results["stage2_flag"] = stage2_flags

----

## Validate That the Stage 3 Rule Sensors Exist

Before I run Stage 3 confirmation logic, I want to verify that the required primary and secondary rule sensors are actually present in the scored dataframe.

This is a quick trust check. If those sensors are missing, then the Stage 3 confirmation logic would be working from an incomplete rule set, which would weaken the final cascade decision.

In [76]:
# --- Stage 3 sanity check: ensure rule sensors exist in scored dataframe
missing_primary = [column for column in stage3_primary_rule_sensors if column not in cascade_results.columns]
missing_secondary = [column for column in stage3_secondary_rule_sensors if column not in cascade_results.columns]

logger.info("Stage3 missing sensors: primary=%d secondary=%d", len(missing_primary), len(missing_secondary))

if missing_primary:
    logger.warning("Missing Stage3 PRIMARY sensors (showing up to 20): %s", missing_primary[:20])
if missing_secondary:
    logger.warning("Missing Stage3 SECONDARY sensors (showing up to 20): %s", missing_secondary[:20])


2026-03-23 23:44:00,766 | INFO | capstone.gold | Stage3 missing sensors: primary=0 secondary=0


----

## Define the Primary Profile Breach Logic

This helper function counts how many Stage 3 primary rule sensors fall outside their reference profile bounds for each row.

The main idea is simple: if enough important primary sensors move outside the stored normal range, that row has stronger evidence that something abnormal is happening.

In [77]:

def compute_primary_breach_count(
    dataframe: pd.DataFrame,
    *,
    reference_profile: pd.DataFrame,
    feature_columns: list[str],
) -> pd.Series:
    reference_lookup = reference_profile.set_index("feature_name")[["lower_bound", "upper_bound"]]

    breach_counts = pd.Series(0, index=dataframe.index, dtype=int)

    for feature_name in feature_columns:
        if feature_name not in dataframe.columns or feature_name not in reference_lookup.index:
            continue

        lower = reference_lookup.loc[feature_name, "lower_bound"]
        upper = reference_lookup.loc[feature_name, "upper_bound"]

        breach_flag = ((dataframe[feature_name] < lower) | (dataframe[feature_name] > upper)).astype(int)
        breach_counts = breach_counts + breach_flag

    breach_counts.name = "stage3_profile_breach_count"
    return breach_counts





## Define the Secondary Corroboration Logic

This helper function does a similar job for the secondary rule sensors.

The difference is that these secondary sensors are used more as corroboration than as the main profile breach signal. In other words, they help support the alert rather than carrying the same priority as the primary sensor group.

In [78]:
cascade_results["stage3_profile_breach_count"] = compute_primary_breach_count(
    cascade_results,
    reference_profile=reference_profile,
    feature_columns=stage3_primary_rule_sensors,
)

----

## Compute the Secondary Breach Count

Here I apply the secondary breach logic to the cascade results dataframe.

This creates the row-level count of secondary rule sensors that move outside their reference profile bounds. Later this gets converted into a corroboration flag that contributes to the Stage 3 evidence count.

In [79]:
def compute_secondary_breach_count(
    dataframe: pd.DataFrame,
    *,
    reference_profile: pd.DataFrame,
    feature_columns: list[str],
) -> pd.Series:
    reference_lookup = reference_profile.set_index("feature_name").to_dict("index")
    breach_counts = pd.Series(0, index=dataframe.index, dtype=int)

    for feature_name in feature_columns:
        if feature_name not in reference_lookup:
            continue

        lower_bound = reference_lookup[feature_name]["lower_bound"]
        upper_bound = reference_lookup[feature_name]["upper_bound"]

        feature_breach_flag = (
            (dataframe[feature_name] < lower_bound) |
            (dataframe[feature_name] > upper_bound)
        ).astype(int)

        breach_counts = breach_counts + feature_breach_flag

    breach_counts.name = "stage3_secondary_breach_count"
    return breach_counts

In [80]:
cascade_results["stage3_secondary_breach_count"] = compute_secondary_breach_count(
    cascade_results,
    reference_profile=reference_profile,
    feature_columns=stage3_secondary_rule_sensors,
)

----

## Define the Persistence Logic

Not every abnormal point matters equally. Some are isolated spikes, while others persist across nearby rows.

This helper function checks for persistence by looking at recent Stage 2 flags inside a rolling window. If enough flags appear within that window, the row receives a persistence flag.

In [81]:
def compute_persistence_flag(
    source_flags: pd.Series,
    *,
    rolling_window_size: int = 3,
    minimum_flags_in_window: int = 2,
) -> pd.Series:
    persistence_flag = (
        source_flags
        .rolling(window=rolling_window_size, min_periods=1)
        .sum()
        .ge(minimum_flags_in_window)
        .astype(int)
    )

    persistence_flag.name = "stage3_persistence_flag"
    return persistence_flag

In [82]:
cascade_results["stage3_persistence_flag"] = compute_persistence_flag(
    cascade_results["stage2_flag"],
    rolling_window_size=STAGE3_ROLLING_WINDOW_SIZE,
    minimum_flags_in_window=STAGE3_MINIMUM_FLAGS_IN_WINDOW,
)


----

## Define the Drift Logic

This helper function checks for rolling drift behavior in the Stage 3 watch features.

The idea is to compare each feature to its recent rolling median and flag rows where the current value drifts far enough away from that recent local behavior. This gives Stage 3 another kind of evidence that is different from simple threshold breaches.

In [83]:
def compute_drift_flag(
    dataframe: pd.DataFrame,
    *,
    feature_columns: list[str],
    rolling_window_size: int = 5,
    drift_threshold_multiplier: float = 1.0,
) -> pd.Series:
    drift_trigger_counts = pd.Series(0, index=dataframe.index, dtype=int)

    for feature_name in feature_columns:
        feature_series = dataframe[feature_name]
        feature_standard_deviation = feature_series.std()

        if pd.isna(feature_standard_deviation) or feature_standard_deviation == 0:
            continue

        rolling_median = feature_series.rolling(window=rolling_window_size, min_periods=1).median()
        rolling_delta = (feature_series - rolling_median).abs()

        feature_drift_flag = (
            rolling_delta > (feature_standard_deviation * drift_threshold_multiplier)
        ).astype(int)

        drift_trigger_counts = drift_trigger_counts + feature_drift_flag

    drift_flag = (drift_trigger_counts >= 1).astype(int)
    drift_flag.name = "stage3_drift_flag"
    return drift_flag

----

## Compute the Drift Flag Inputs

Here I build the combined Stage 3 watch feature list that will be used for the drift check.

This keeps the drift logic focused on the same important sensor groups already being watched by the Stage 3 rule layer.

In [84]:
stage3_rule_watch_features = list(dict.fromkeys(
    stage3_primary_rule_sensors + stage3_secondary_rule_sensors
))

cascade_results["stage3_drift_flag"] = compute_drift_flag(
    cascade_results,
    feature_columns=stage3_rule_watch_features,
    rolling_window_size=STAGE3_DRIFT_ROLLING_WINDOW_SIZE,
    drift_threshold_multiplier=STAGE3_DRIFT_THRESHOLD_MULTIPLIER,
)

----

## Build the Final Stage 3 Evidence Flags and Final Cascade Decision

Now I combine the Stage 3 evidence into the final confirmation logic.

This section creates:
- the primary profile breach flag
- the secondary corroboration flag
- the persistence flag
- the drift flag
- the overall Stage 3 evidence count
- the final `cascade_final_flag`

The final cascade decision is intentionally strict. A row must first pass through Stage 1 and Stage 2, and then it must also show enough Stage 3 evidence to be considered a final cascade alert.

In [85]:
cascade_results["stage3_profile_breach_flag"] = (
    cascade_results["stage3_profile_breach_count"] >= STAGE3_MIN_PRIMARY_SENSOR_HITS
).astype(int)

cascade_results["stage3_strong_primary_flag"] = (
    cascade_results["stage3_profile_breach_count"] >= STAGE3_STRONG_PRIMARY_HITS
).astype(int)

cascade_results["stage3_corroboration_flag"] = (
    cascade_results["stage3_secondary_breach_count"] >= STAGE3_MIN_SECONDARY_SENSOR_HITS
).astype(int)

cascade_results["stage3_rule_evidence_count"] = (
    cascade_results["stage3_profile_breach_flag"]
    + cascade_results["stage3_persistence_flag"]
    + cascade_results["stage3_drift_flag"]
    + cascade_results["stage3_corroboration_flag"]
)

cascade_results["stage3_weighted_evidence_score"] = (
    cascade_results["stage3_profile_breach_flag"] * STAGE3_PROFILE_BREACH_WEIGHT
    + cascade_results["stage3_corroboration_flag"] * STAGE3_CORROBORATION_WEIGHT
    + cascade_results["stage3_persistence_flag"] * STAGE3_PERSISTENCE_WEIGHT
    + cascade_results["stage3_drift_flag"] * STAGE3_DRIFT_WEIGHT
)

# ---------------------------------
# Stage 3 becomes confirmation logic
# ---------------------------------

cascade_results["stage3_confirmed_flag"] = (
    (cascade_results["stage3_strong_primary_flag"] == 1)
    | (
        (cascade_results["stage3_profile_breach_flag"] == 1)
        & (
            cascade_results["stage3_weighted_evidence_score"]
            >= STAGE3_MIN_WEIGHTED_SCORE
        )
    )
).astype(int)

# Keep tuned Stage 2 as the main final model
cascade_results["cascade_final_flag"] = cascade_results["stage2_flag"].astype(int)

# Optional Stage 3 filter variants for later comparison
cascade_results["cascade_stage3_relaxed_flag"] = (
    (cascade_results["stage2_flag"] == 1)
    & (
        (cascade_results["stage3_profile_breach_flag"] == 1)
        | (cascade_results["stage3_corroboration_flag"] == 1)
    )
).astype(int)

cascade_results["cascade_stage3_medium_flag"] = (
    (cascade_results["stage2_flag"] == 1)
    & (
        (cascade_results["stage3_confirmed_flag"] == 1)
        | (cascade_results["stage3_rule_evidence_count"] >= 2)
    )
).astype(int)

cascade_results["cascade_stage3_strict_flag"] = (
    (cascade_results["stage2_flag"] == 1)
    & (cascade_results["stage3_strong_primary_flag"] == 1)
).astype(int)

# Priority labels for operational triage
cascade_results["alert_priority"] = np.select(
    [
        (cascade_results["stage2_flag"] == 1) & (cascade_results["stage3_strong_primary_flag"] == 1),
        (cascade_results["stage2_flag"] == 1) & (cascade_results["stage3_confirmed_flag"] == 1),
        (cascade_results["stage2_flag"] == 1),
    ],
    [
        "high",
        "medium",
        "low",
    ],
    default="none",
)

stage3_summary = {
    "stage3_variant": "confirmation_and_priority_layer",
    "primary_rule_sensor_count": int(len(stage3_primary_rule_sensors)),
    "secondary_rule_sensor_count": int(len(stage3_secondary_rule_sensors)),
    "strong_primary_rows_all": int((cascade_results["stage3_strong_primary_flag"] == 1).sum()),
    "strong_primary_rows_test": int(cascade_results.loc[test_mask, "stage3_strong_primary_flag"].sum()),
    "profile_breach_rows_all": int((cascade_results["stage3_profile_breach_flag"] == 1).sum()),
    "profile_breach_rows_test": int(cascade_results.loc[test_mask, "stage3_profile_breach_flag"].sum()),
    "corroboration_rows_all": int((cascade_results["stage3_corroboration_flag"] == 1).sum()),
    "corroboration_rows_test": int(cascade_results.loc[test_mask, "stage3_corroboration_flag"].sum()),
    "persistence_rows_all": int((cascade_results["stage3_persistence_flag"] == 1).sum()),
    "persistence_rows_test": int(cascade_results.loc[test_mask, "stage3_persistence_flag"].sum()),
    "drift_rows_all": int((cascade_results["stage3_drift_flag"] == 1).sum()),
    "drift_rows_test": int(cascade_results.loc[test_mask, "stage3_drift_flag"].sum()),
    "weighted_score_mean_all": float(cascade_results["stage3_weighted_evidence_score"].mean()),
    "weighted_score_mean_test": float(cascade_results.loc[test_mask, "stage3_weighted_evidence_score"].mean()),
    "confirmed_rows_all": int((cascade_results["stage3_confirmed_flag"] == 1).sum()),
    "confirmed_rows_test": int(cascade_results.loc[test_mask, "stage3_confirmed_flag"].sum()),
    "final_tuned_alert_count_all_rows": int(cascade_results["cascade_final_flag"].sum()),
    "final_tuned_alert_count_test_rows": int(cascade_results.loc[test_mask, "cascade_final_flag"].sum()),
    "stage3_relaxed_alert_count_all_rows": int(cascade_results["cascade_stage3_relaxed_flag"].sum()),
    "stage3_relaxed_alert_count_test_rows": int(cascade_results.loc[test_mask, "cascade_stage3_relaxed_flag"].sum()),
    "stage3_medium_alert_count_all_rows": int(cascade_results["cascade_stage3_medium_flag"].sum()),
    "stage3_medium_alert_count_test_rows": int(cascade_results.loc[test_mask, "cascade_stage3_medium_flag"].sum()),
    "stage3_strict_alert_count_all_rows": int(cascade_results["cascade_stage3_strict_flag"].sum()),
    "stage3_strict_alert_count_test_rows": int(cascade_results.loc[test_mask, "cascade_stage3_strict_flag"].sum()),
    "high_priority_rows_all": int((cascade_results["alert_priority"] == "high").sum()),
    "high_priority_rows_test": int((cascade_results.loc[test_mask, "alert_priority"] == "high").sum()),
    "medium_priority_rows_all": int((cascade_results["alert_priority"] == "medium").sum()),
    "medium_priority_rows_test": int((cascade_results.loc[test_mask, "alert_priority"] == "medium").sum()),
    "low_priority_rows_all": int((cascade_results["alert_priority"] == "low").sum()),
    "low_priority_rows_test": int((cascade_results.loc[test_mask, "alert_priority"] == "low").sum()),
}

ledger.add(
    kind="step",
    step="run_cascade_stage3_confirmation",
    message="Applied Stage 3 confirmation, priority labeling, and optional filter variants to all rows of the scaled dataset.",
    data=stage3_summary,
    logger=logger,
)

cascade_results.head(5)

2026-03-23 23:44:04,742 | INFO | capstone.gold | LEDGER | {'ts_utc': '2026-03-23T23:44:04.742137+00:00', 'stage': 'gold', 'recipe': 'gold_modeling__v001_cascade', 'kind': 'step', 'step': 'run_cascade_stage3_confirmation', 'message': 'Applied Stage 3 confirmation, priority labeling, and optional filter variants to all rows of the scaled dataset.', 'why': None, 'consequence': None, 'data': {'stage3_variant': 'confirmation_and_priority_layer', 'primary_rule_sensor_count': 5, 'secondary_rule_sensor_count': 7, 'strong_primary_rows_all': 12044, 'strong_primary_rows_test': 6891, 'profile_breach_rows_all': 72702, 'profile_breach_rows_test': 49472, 'corroboration_rows_all': 150151, 'corroboration_rows_test': 80276, 'persistence_rows_all': 13875, 'persistence_rows_test': 642, 'drift_rows_all': 2448, 'drift_rows_test': 1409, 'weighted_score_mean_all': 1.415568264342774, 'weighted_score_mean_test': 2.160843495571529, 'confirmed_rows_all': 68782, 'confirmed_rows_test': 48006, 'final_tuned_alert_cou

,meta__asset_id,meta__dataset,meta__episode_id,meta__event_id,meta__ingested_at_utc,meta__parent_truth_hash,meta__pipeline_mode,meta__record_id,meta__run_id,meta__source_file,meta__source_row_id,meta__split,meta__truth_hash,event_time,event_step,time_index,event_date,anomaly_flag,is_anomaly,is_normal,status_normal_value,sensor_00,sensor_01,sensor_02,sensor_03,sensor_04,sensor_05,sensor_06,sensor_07,sensor_08,sensor_09,sensor_10,sensor_11,sensor_12,sensor_13,sensor_14,sensor_16,sensor_17,sensor_18,sensor_19,sensor_20,sensor_21,sensor_22,sensor_23,sensor_24,sensor_25,sensor_26,sensor_27,sensor_28,sensor_29,sensor_30,sensor_31,sensor_32,sensor_33,sensor_34,sensor_35,sensor_36,sensor_37,sensor_38,sensor_39,sensor_40,sensor_41,sensor_42,sensor_43,sensor_44,sensor_45,sensor_46,sensor_47,sensor_48,sensor_49,sensor_51,timestamp,machine_status,meta__is_train_flag,stage1_score,stage1_flag,stage2_score,stage2_raw_flag,stage2_flag,stage3_profile_breach_count,stage3_secondary_breach_count,stage3_persistence_flag,stage3_drift_flag,stage3_profile_breach_flag,stage3_strong_primary_flag,stage3_corroboration_flag,stage3_rule_evidence_count,stage3_weighted_evidence_score,stage3_confirmed_flag,cascade_final_flag,cascade_stage3_relaxed_flag,cascade_stage3_medium_flag,cascade_stage3_strict_flag,alert_priority
0,asset__001,pump,0,pump:asset__001:run__001:0,2026-03-22 00:55:01.594555+00:00,150ec91684fac04d2ee1c41c1b9297bde4fcd92fa30a2f...,batch,14598431322315673869,run__001,sensor.csv,0,train,8a62f44686a717446607a3d0303285188fa60d19339eef...,2018-04-01 00:00:00+00:00,0,0,2018-04-01 00:00:00+00:00,0,0,1,NORMAL,0.232560,-0.509807,0.537036,1.090905,0.227271,-0.144184,-0.212771,0.000000,0.633343,-0.133358,-0.966540,0.663285,-0.099631,-0.152932,0.001485,-0.017980,0.259816,0.203007,0.012069,0.001266,0.026039,-0.004667,0.375041,0.470634,0.054448,0.018867,-0.398656,-0.746314,-0.137762,-0.554838,-1.070844,-0.832376,-0.653530,-0.038647,-0.157592,-0.223411,0.170945,-1.156251,-0.909090,0.527273,-0.909090,-0.882353,-0.125000,0.000000,4.000000,0.928571,-0.608697,0.715953,1.900001,0.014598,2018-04-01 00:00:00,NORMAL,True,0.391431,0,0.420189,0,0,0,1,0,0,0,0,1,1,1.0,0,0,0,0,0,none
1,asset__001,pump,0,pump:asset__001:run__001:1,2026-03-22 00:55:01.594555+00:00,150ec91684fac04d2ee1c41c1b9297bde4fcd92fa30a2f...,batch,15954729095895098000,run__001,sensor.csv,1,train,8a62f44686a717446607a3d0303285188fa60d19339eef...,2018-04-01 00:01:00+00:00,1,1,2018-04-01 00:00:00+00:00,0,0,1,NORMAL,0.232560,-0.509807,0.537036,1.090905,0.227271,-0.144184,-0.212771,0.000000,0.633343,-0.133358,-0.966540,0.663285,-0.099631,-0.152932,0.001485,-0.017980,0.259816,0.203007,0.012069,0.001266,0.026039,-0.004667,0.375041,0.470634,0.054448,0.018867,-0.398656,-0.746314,-0.137762,-0.554838,-1.070844,-0.832376,-0.653530,-0.038647,-0.157592,-0.223411,0.170945,-1.156251,-0.909090,0.527273,-0.909090,-0.882353,-0.125000,0.000000,4.000000,0.928571,-0.608697,0.715953,1.900001,0.014598,2018-04-01 00:01:00,NORMAL,True,0.391431,0,0.420189,0,0,0,1,0,0,0,0,1,1,1.0,0,0,0,0,0,none
2,asset__001,pump,0,pump:asset__001:run__001:2,2026-03-22 00:55:01.594555+00:00,150ec91684fac04d2ee1c41c1b9297bde4fcd92fa30a2f...,batch,10041703297090838359,run__001,sensor.csv,2,train,8a62f44686a717446607a3d0303285188fa60d19339eef...,2018-04-01 00:02:00+00:00,2,2,2018-04-01 00:00:00+00:00,0,0,1,NORMAL,-0.255821,-0.392159,0.537036,1.127271,0.670453,-0.485083,-0.468102,-0.185694,0.750017,-0.333349,-0.869635,0.742744,0.084552,-0.140848,0.063680,0.024252,-0.023257,-0.051481,0.031807,0.039803,0.034031,0.045869,0.539582,0.559032,0.045512,0.031031,-0.029962,-0.769896,-0.025671,0.380643,-0.866485,-0.748481,-0.586075,-0.053782,-0.147948,-0.214093,0.276319,-1.031250,-0.954545,0.454546,-0.999999,-0.882353,-0.166667,-0.045454,3.954546,0.964285,-0.608696,0.688715,1.833334,0.072992,2018-04-01 00:02:00,NORMAL,True,0.384923,0,0.411562,0,0,0,0,0,0,0,0,0,0,0.0,0,0,0,0,0,none
3,asset__001,pump,0,pump:asset__001:run__001:3,2026-03-22 00:55:01.594555+00:00,

### Ask

How should I think about the final Stage 3 rule?

### Answer

Stage 3 is the confirmation layer that tries to reduce weak or isolated model alerts.

In this notebook, a row can become a final cascade alert in one of two main ways:
- it shows enough primary profile breaches on its own
- or it builds enough combined rule evidence across the Stage 3 checks

So Stage 3 is not replacing the model stages. It is acting as a final confirmation layer after the row has already passed through both model filters.

----

## Build the Main Cascade Metrics

Here I summarize the main tuned cascade results.

This includes:
- Stage 1 alert counts
- Stage 2 alert counts
- final cascade alert counts
- test-row alert counts
- evaluation metrics such as precision, recall, and F1 when test labels are available

This is the main performance snapshot for the tuned cascade notebook.

In [86]:
cascade_metrics = {
    "model": "3-Stage Cascade",
    "stage1_alert_count_all_rows": int(cascade_results["stage1_flag"].sum()),
    "stage2_alert_count_all_rows": int(cascade_results["stage2_flag"].sum()),
    "final_alert_count_all_rows": int(cascade_results["cascade_final_flag"].sum()),
    "stage1_alert_count_test_rows": int(cascade_results.loc[test_mask, "stage1_flag"].sum()),
    "stage2_alert_count_test_rows": int(cascade_results.loc[test_mask, "stage2_flag"].sum()),
    "final_alert_count_test_rows": int(cascade_results.loc[test_mask, "cascade_final_flag"].sum()),
    "stage3_relaxed_alert_count_test_rows": int(cascade_results.loc[test_mask, "cascade_stage3_relaxed_flag"].sum()),
    "stage3_medium_alert_count_test_rows": int(cascade_results.loc[test_mask, "cascade_stage3_medium_flag"].sum()),
    "stage3_strict_alert_count_test_rows": int(cascade_results.loc[test_mask, "cascade_stage3_strict_flag"].sum()),
}

if test_labels is not None:
    cascade_test_flags = (
        cascade_results
        .loc[test_mask, "cascade_final_flag"]
        .astype(int)
        .values
    )

    precision, recall, f1, _ = precision_recall_fscore_support(
        test_labels,
        cascade_test_flags,
        average="binary",
        zero_division=0,
    )

    cascade_metrics["precision"] = float(precision)
    cascade_metrics["recall"] = float(recall)
    cascade_metrics["f1"] = float(f1)

    for variant_name, column_name in [
        ("stage3_relaxed", "cascade_stage3_relaxed_flag"),
        ("stage3_medium", "cascade_stage3_medium_flag"),
        ("stage3_strict", "cascade_stage3_strict_flag"),
    ]:
        variant_test_flags = (
            cascade_results
            .loc[test_mask, column_name]
            .astype(int)
            .values
        )

        v_precision, v_recall, v_f1, _ = precision_recall_fscore_support(
            test_labels,
            variant_test_flags,
            average="binary",
            zero_division=0,
        )

        cascade_metrics[f"{variant_name}_precision"] = float(v_precision)
        cascade_metrics[f"{variant_name}_recall"] = float(v_recall)
        cascade_metrics[f"{variant_name}_f1"] = float(v_f1)

cascade_metrics

{'model': '3-Stage Cascade',
 'stage1_alert_count_all_rows': 22742,
 'stage2_alert_count_all_rows': 13898,
 'final_alert_count_all_rows': 13898,
 'stage1_alert_count_test_rows': 2288,
 'stage2_alert_count_test_rows': 648,
 'final_alert_count_test_rows': 648,
 'stage3_relaxed_alert_count_test_rows': 648,
 'stage3_medium_alert_count_test_rows': 646,
 'stage3_strict_alert_count_test_rows': 62,
 'precision': 0.09259259259259259,
 'recall': 0.5084745762711864,
 'f1': 0.1566579634464752,
 'stage3_relaxed_precision': 0.09259259259259259,
 'stage3_relaxed_recall': 0.5084745762711864,
 'stage3_relaxed_f1': 0.1566579634464752,
 'stage3_medium_precision': 0.09287925696594428,
 'stage3_medium_recall': 0.5084745762711864,
 'stage3_medium_f1': 0.15706806282722513,
 'stage3_strict_precision': 0.3870967741935484,
 'stage3_strict_recall': 0.2033898305084746,
 'stage3_strict_f1': 0.26666666666666666}

### Ask

What should I pay attention to in the tuned cascade metrics?

### Answer

The main things I care about are:
- how many rows survive from Stage 1 into Stage 2
- how many rows survive from Stage 2 into the final cascade output
- whether the tuned Stage 2 branch changed the final alert set in a useful way
- precision, recall, and F1 on the test rows when labels are available

For this notebook, the key question is not just whether the cascade detects anomalies. It is whether the tuned Stage 2 search helps produce a cleaner final alert set than the default cascade and the baseline.

----

## Build the Cascade Summary, Threshold Records, and Truth Artifact

Now I convert the tuned cascade results into formal pipeline artifacts.

This section does several important things:
- summarizes the cascade thresholds and metrics
- records the tuned Stage 2 selection details
- builds the main cascade summary record
- creates the Gold cascade truth record
- stores runtime facts and artifact paths
- links the cascade output back to the parent Gold truth
- prepares the final saved deliverables for downstream comparison

At this point, the notebook output becomes more than just a scored dataframe. It becomes a tracked modeling artifact in the pipeline.

In [87]:
stage1_alert_count_all_rows = int(cascade_results["stage1_flag"].sum())
stage2_alert_count_all_rows = int(cascade_results["stage2_flag"].sum())
final_cascade_alert_count_all_rows = int(cascade_results["cascade_final_flag"].sum())

stage1_alert_count_test_rows = int(cascade_results.loc[test_mask, "stage1_flag"].sum())
stage2_alert_count_test_rows = int(cascade_results.loc[test_mask, "stage2_flag"].sum())
final_cascade_alert_count_test_rows = int(cascade_results.loc[test_mask, "cascade_final_flag"].sum())

stage3_relaxed_alert_count_all_rows = int(cascade_results["cascade_stage3_relaxed_flag"].sum())
stage3_medium_alert_count_all_rows = int(cascade_results["cascade_stage3_medium_flag"].sum())
stage3_strict_alert_count_all_rows = int(cascade_results["cascade_stage3_strict_flag"].sum())

stage3_relaxed_alert_count_test_rows = int(cascade_results.loc[test_mask, "cascade_stage3_relaxed_flag"].sum())
stage3_medium_alert_count_test_rows = int(cascade_results.loc[test_mask, "cascade_stage3_medium_flag"].sum())
stage3_strict_alert_count_test_rows = int(cascade_results.loc[test_mask, "cascade_stage3_strict_flag"].sum())

cascade_thresholds = {
    "cascade_variant": CASCADE_VARIANT,
    "stage1_threshold_percentile": float(STAGE1_THRESHOLD_PERCENTILE),
    "stage1_threshold": float(stage1_threshold),
    "stage2_selection_mode": STAGE2_SELECTION_MODE,
    "stage2_selected_threshold_percentile": float(stage2_selected_threshold_percentile),
    "stage2_threshold": float(stage2_threshold),
    "stage2_best_params": stage2_best_params,
    "stage3_variant": "confirmation_and_priority_layer",
    "stage3_strong_primary_hits": int(STAGE3_STRONG_PRIMARY_HITS),
    "stage3_min_primary_sensor_hits": int(STAGE3_MIN_PRIMARY_SENSOR_HITS),
    "stage3_min_secondary_sensor_hits": int(STAGE3_MIN_SECONDARY_SENSOR_HITS),
    "stage3_profile_breach_weight": float(STAGE3_PROFILE_BREACH_WEIGHT),
    "stage3_corroboration_weight": float(STAGE3_CORROBORATION_WEIGHT),
    "stage3_persistence_weight": float(STAGE3_PERSISTENCE_WEIGHT),
    "stage3_drift_weight": float(STAGE3_DRIFT_WEIGHT),
    "stage3_min_weighted_score": float(STAGE3_MIN_WEIGHTED_SCORE),
    "stage3_rolling_window_size": int(STAGE3_ROLLING_WINDOW_SIZE),
    "stage3_minimum_flags_in_window": int(STAGE3_MINIMUM_FLAGS_IN_WINDOW),
    "stage3_drift_rolling_window_size": int(STAGE3_DRIFT_ROLLING_WINDOW_SIZE),
    "stage3_drift_threshold_multiplier": float(STAGE3_DRIFT_THRESHOLD_MULTIPLIER),
}

cascade_summary = {
    "dataset_name": DATASET_NAME,
    "cascade_variant": CASCADE_VARIANT,
    "stage3_variant": "confirmation_and_priority_layer",
    "cascade_metrics": cascade_metrics,
    "stage1_alert_count_all_rows": stage1_alert_count_all_rows,
    "stage2_alert_count_all_rows": stage2_alert_count_all_rows,
    "final_cascade_alert_count_all_rows": final_cascade_alert_count_all_rows,
    "stage1_alert_count_test_rows": stage1_alert_count_test_rows,
    "stage2_alert_count_test_rows": stage2_alert_count_test_rows,
    "final_cascade_alert_count_test_rows": final_cascade_alert_count_test_rows,
    "stage3_relaxed_alert_count_all_rows": stage3_relaxed_alert_count_all_rows,
    "stage3_medium_alert_count_all_rows": stage3_medium_alert_count_all_rows,
    "stage3_strict_alert_count_all_rows": stage3_strict_alert_count_all_rows,
    "stage3_relaxed_alert_count_test_rows": stage3_relaxed_alert_count_test_rows,
    "stage3_medium_alert_count_test_rows": stage3_medium_alert_count_test_rows,
    "stage3_strict_alert_count_test_rows": stage3_strict_alert_count_test_rows,
    "result_row_count": int(len(cascade_results)),
    "stage1_feature_count": int(len(stage1_feature_columns)),
    "stage2_feature_count": int(len(stage2_feature_columns)),
    "stage3_primary_rule_count": int(len(stage3_primary_rule_sensors)),
    "stage3_secondary_rule_count": int(len(stage3_secondary_rule_sensors)),
    "stage2_best_params": stage2_best_params,
    "stage2_search_candidate_count": int(len(stage2_search_results)),
    "stage2_selection_mode": STAGE2_SELECTION_MODE,
    "stage3_strong_primary_hits": int(STAGE3_STRONG_PRIMARY_HITS),
    "stage3_min_weighted_score": float(STAGE3_MIN_WEIGHTED_SCORE),
    "stage3_summary": stage3_summary,
}

truth_config_snapshot = (
    TRUTH_CONFIG
    if "TRUTH_CONFIG" in globals()
    else {
        "runtime": {
            "stage": "gold_cascade",
            "dataset": DATASET_NAME,
            "cascade_variant": CASCADE_VARIANT,
            "mode": RUN_MODE if "RUN_MODE" in globals() else None,
            "profile": CONFIG_PROFILE if "CONFIG_PROFILE" in globals() else "default",
        }
    }
)

cascade_truth_layer_name = "gold_cascade"
cascade_process_run_id = (
    GOLD_PROCESS_RUN_ID
    if "GOLD_PROCESS_RUN_ID" in globals()
    else make_process_run_id("gold_cascade_process")
)

cascade_truth = initialize_layer_truth(
    truth_version=TRUTH_VERSION,
    dataset_name=DATASET_NAME,
    layer_name=cascade_truth_layer_name,
    process_run_id=cascade_process_run_id,
    pipeline_mode=PIPELINE_MODE,
    parent_truth_hash=GOLD_PARENT_TRUTH_HASH,
)

cascade_truth = update_truth_section(
    cascade_truth,
    "config_snapshot",
    truth_config_snapshot,
)

cascade_truth = update_truth_section(
    cascade_truth,
    "runtime_facts",
    {
        "cascade_variant": CASCADE_VARIANT,
        "stage3_variant": "confirmation_and_priority_layer",
        "result_row_count": int(len(cascade_results)),
        "stage1_alert_count_all_rows": stage1_alert_count_all_rows,
        "stage2_alert_count_all_rows": stage2_alert_count_all_rows,
        "final_cascade_alert_count_all_rows": final_cascade_alert_count_all_rows,
        "stage1_alert_count_test_rows": stage1_alert_count_test_rows,
        "stage2_alert_count_test_rows": stage2_alert_count_test_rows,
        "final_cascade_alert_count_test_rows": final_cascade_alert_count_test_rows,
        "stage3_relaxed_alert_count_test_rows": stage3_relaxed_alert_count_test_rows,
        "stage3_medium_alert_count_test_rows": stage3_medium_alert_count_test_rows,
        "stage3_strict_alert_count_test_rows": stage3_strict_alert_count_test_rows,
        "stage1_feature_count": int(len(stage1_feature_columns)),
        "stage2_feature_count": int(len(stage2_feature_columns)),
        "stage3_primary_rule_count": int(len(stage3_primary_rule_sensors)),
        "stage3_secondary_rule_count": int(len(stage3_secondary_rule_sensors)),
    },
)

cascade_truth = update_truth_section(
    cascade_truth,
    "artifact_paths",
    {
        "cascade_variant": CASCADE_VARIANT,
        "gold_truth_path": str(GOLD_TRUTH_PATH),
        "gold_fit_path": str(GOLD_FIT_DATA_PATH),
        "gold_scored_path": str(GOLD_PREPROCESSED_SCALED_DATA_PATH),
        "stage1_features_path": str(STAGE1_FEATURES_PATH),
        "stage2_features_path": str(STAGE2_FEATURES_PATH),
        "stage3_primary_path": str(STAGE3_PRIMARY_PATH),
        "stage3_secondary_path": str(STAGE3_SECONDARY_PATH),
        "cascade_tuned_results_path_csv": str(CASCADE_RESULTS_PATH_CSV),
        "cascade_tuned_results_path_pickle": str(CASCADE_RESULTS_PATH_PICKLE),
        "cascade_tuned_stage1_model_artifact_path": str(STAGE1_MODEL_ARTIFACT_PATH),
        "cascade_tuned_stage1_models_path": str(STAGE1_MODELS_PATH),
        "cascade_tuned_stage2_model_artifact_path": str(STAGE2_MODEL_ARTIFACT_PATH),
        "cascade_tuned_stage2_models_path": str(STAGE2_MODELS_PATH),
        "cascade_tuned_thresholds_path": str(CASCADE_THRESHOLDS_PATH),
        "cascade_tuned_summary_path": str(CASCADE_SUMMARY_PATH),
        "cascade_tuned_metadata_path": str(CASCADE_METADATA_PATH),
        "cascade_tuned_reference_profile_path": str(CASCADE_REFERENCE_PROFILE_PATH),
    },
)

cascade_meta_columns = sorted(
    set(
        identify_meta_columns(cascade_results)
        + [
            "meta__truth_hash",
            "meta__parent_truth_hash",
            "meta__pipeline_mode",
        ]
    )
)

cascade_feature_columns = identify_feature_columns(cascade_results)

cascade_truth_record = build_truth_record(
    truth_base=cascade_truth,
    row_count=len(cascade_results),
    column_count=cascade_results.shape[1] + 3,
    meta_columns=cascade_meta_columns,
    feature_columns=cascade_feature_columns,
)

CASCADE_TRUTH_HASH = cascade_truth_record["truth_hash"]

cascade_results = stamp_truth_columns(
    cascade_results,
    truth_hash=CASCADE_TRUTH_HASH,
    parent_truth_hash=GOLD_PARENT_TRUTH_HASH,
    pipeline_mode=PIPELINE_MODE,
)

cascade_truth_path = save_truth_record(
    cascade_truth_record,
    truth_dir=TRUTHS_PATH,
    dataset_name=DATASET_NAME,
    layer_name=cascade_truth_layer_name,
)

append_truth_index(
    cascade_truth_record,
    truth_index_path=TRUTH_INDEX_PATH,
)

cascade_summary["cascade_truth_hash"] = CASCADE_TRUTH_HASH
cascade_summary["cascade_truth_path"] = str(cascade_truth_path)
cascade_summary["cascade_process_run_id"] = cascade_process_run_id
cascade_summary["gold_truth_hash"] = GOLD_PARENT_TRUTH_HASH
cascade_summary["gold_truth_path"] = str(GOLD_TRUTH_PATH)
cascade_summary["gold_process_run_id"] = gold_truth.get("process_run_id")
cascade_summary["gold_feature_set_id"] = gold_truth_runtime_facts.get("feature_set_id")

cascade_metadata = {
    "cascade_variant": CASCADE_VARIANT,
    "stage2_selection_mode": STAGE2_SELECTION_MODE,
    "cascade_tuned_results_path_csv": str(CASCADE_RESULTS_PATH_CSV),
    "cascade_tuned_results_path_pickle": str(CASCADE_RESULTS_PATH_PICKLE),
    "cascade_tuned_stage1_model_artifact_path": str(STAGE1_MODEL_ARTIFACT_PATH),
    "cascade_tuned_stage1_models_path": str(STAGE1_MODELS_PATH),
    "cascade_tuned_stage2_model_artifact_path": str(STAGE2_MODEL_ARTIFACT_PATH),
    "cascade_tuned_stage2_models_path": str(STAGE2_MODELS_PATH),
    "cascade_tuned_thresholds_path": str(CASCADE_THRESHOLDS_PATH),
    "cascade_tuned_summary_path": str(CASCADE_SUMMARY_PATH),
    "cascade_tuned_metadata_path": str(CASCADE_METADATA_PATH),
    "cascade_tuned_reference_profile_path": str(CASCADE_REFERENCE_PROFILE_PATH),
    "gold_fit_path": str(GOLD_FIT_DATA_PATH),
    "gold_scored_path": str(GOLD_PREPROCESSED_SCALED_DATA_PATH),

    # upstream truth linkage
    "gold_truth_hash": GOLD_PARENT_TRUTH_HASH,
    "gold_truth_path": str(GOLD_TRUTH_PATH),
    "gold_process_run_id": gold_truth.get("process_run_id"),
    "gold_feature_set_id": gold_truth_runtime_facts.get("feature_set_id"),
    "gold_scaler_path": gold_truth_artifact_paths.get("scaler_path"),
    "gold_scaler_kind": gold_truth_runtime_facts.get("scaler_kind_runtime"),
    "gold_recommended_imputation": gold_truth_runtime_facts.get("recommended_imputation"),

    # stage truth linkage
    "cascade_truth_hash": CASCADE_TRUTH_HASH,
    "cascade_truth_path": str(cascade_truth_path),
    "cascade_process_run_id": cascade_process_run_id,
    "stage3_variant": "confirmation_and_priority_layer",
}

cascade_results.to_csv(CASCADE_RESULTS_PATH_CSV, index=False)
cascade_results.to_pickle(CASCADE_RESULTS_PATH_PICKLE)

reference_profile.to_csv(CASCADE_REFERENCE_PROFILE_PATH, index=False)

joblib.dump(stage1_model, STAGE1_MODEL_ARTIFACT_PATH)
joblib.dump(stage1_model, STAGE1_MODELS_PATH)

joblib.dump(stage2_model, STAGE2_MODEL_ARTIFACT_PATH)
joblib.dump(stage2_model, STAGE2_MODELS_PATH)

save_json(cascade_thresholds, CASCADE_THRESHOLDS_PATH)
save_json(cascade_summary, CASCADE_SUMMARY_PATH)
save_json(cascade_metadata, CASCADE_METADATA_PATH)

wandb.save(str(CASCADE_RESULTS_PATH_CSV))
wandb.save(str(CASCADE_RESULTS_PATH_PICKLE))
wandb.save(str(CASCADE_REFERENCE_PROFILE_PATH))
wandb.save(str(STAGE1_MODEL_ARTIFACT_PATH))
wandb.save(str(STAGE1_MODELS_PATH))
wandb.save(str(STAGE2_MODEL_ARTIFACT_PATH))
wandb.save(str(STAGE2_MODELS_PATH))
wandb.save(str(CASCADE_THRESHOLDS_PATH))
wandb.save(str(CASCADE_SUMMARY_PATH))
wandb.save(str(CASCADE_METADATA_PATH))
wandb.save(str(cascade_truth_path))

ledger.add(
    kind="step",
    step="save_cascade_outputs",
    message="Saved cascade results, trained Stage 1 and Stage 2 models, thresholds, summary, metadata, reference profile, and cascade stage truth record.",
    data={
        "cascade_tuned_results_path_csv": str(CASCADE_RESULTS_PATH_CSV),
        "cascade_tuned_results_path_pickle": str(CASCADE_RESULTS_PATH_PICKLE),
        "cascade_tune_reference_profile_path": str(CASCADE_REFERENCE_PROFILE_PATH),
        "cascade_tuned_stage1_model_artifact_path": str(STAGE1_MODEL_ARTIFACT_PATH),
        "cascade_tuned_stage1_models_path": str(STAGE1_MODELS_PATH),
        "cascade_tuned_stage2_model_artifact_path": str(STAGE2_MODEL_ARTIFACT_PATH),
        "cascade_tuned_stage2_models_path": str(STAGE2_MODELS_PATH),
        "cascade_tuned_thresholds_path": str(CASCADE_THRESHOLDS_PATH),
        "cascade_tuned_summary_path": str(CASCADE_SUMMARY_PATH),
        "cascade_tuned_metadata_path": str(CASCADE_METADATA_PATH),
        "cascade_truth_hash": CASCADE_TRUTH_HASH,
        "cascade_truth_path": str(cascade_truth_path),
        "result_row_count": int(len(cascade_results)),
        "final_alert_count_all_rows": final_cascade_alert_count_all_rows,
        "final_alert_count_test_rows": final_cascade_alert_count_test_rows,
        "stage3_relaxed_alert_count_test_rows": stage3_relaxed_alert_count_test_rows,
        "stage3_medium_alert_count_test_rows": stage3_medium_alert_count_test_rows,
        "stage3_strict_alert_count_test_rows": stage3_strict_alert_count_test_rows,
    },
    logger=logger,
)

2026-03-23 23:45:08,629 | INFO | capstone.file_io | Saved JSON: /workspace/artifacts/gold/pump/pump__gold__cascade_stage3_improved_thresholds.json
2026-03-23 23:45:08,656 | INFO | capstone.file_io | Saved JSON: /workspace/artifacts/gold/pump/pump__gold__cascade_stage3_improved_summary.json
2026-03-23 23:45:08,683 | INFO | capstone.file_io | Saved JSON: /workspace/artifacts/gold/pump/pump__gold__cascade_stage3_improved_metadata.json
2026-03-23 23:45:09,180 | INFO | capstone.gold | LEDGER | {'ts_utc': '2026-03-23T23:45:09.180849+00:00', 'stage': 'gold', 'recipe': 'gold_modeling__v001_cascade', 'kind': 'step', 'step': 'save_cascade_outputs', 'message': 'Saved cascade results, trained Stage 1 and Stage 2 models, thresholds, summary, metadata, reference profile, and cascade stage truth record.', 'why': None, 'consequence': None, 'data': {'cascade_tuned_results_path_csv': '/workspace/artifacts/gold/pump/pump__gold__cascade_stage3_improved_results.csv', 'cascade_tuned_results_path_pickle': '/

{'ts_utc': '2026-03-23T23:45:09.180849+00:00',
 'stage': 'gold',
 'recipe': 'gold_modeling__v001_cascade',
 'kind': 'step',
 'step': 'save_cascade_outputs',
 'message': 'Saved cascade results, trained Stage 1 and Stage 2 models, thresholds, summary, metadata, reference profile, and cascade stage truth record.',
 'why': None,
 'consequence': None,
 'data': {'cascade_tuned_results_path_csv': '/workspace/artifacts/gold/pump/pump__gold__cascade_stage3_improved_results.csv',
  'cascade_tuned_results_path_pickle': '/workspace/artifacts/gold/pump/pump__gold__cascade_stage3_improved_results.pkl',
  'cascade_tune_reference_profile_path': '/workspace/artifacts/gold/pump/pump__gold__cascade_tuned_reference_profile.csv',
  'cascade_tuned_stage1_model_artifact_path': '/workspace/artifacts/gold/pump/pump__gold__cascade_stage3_improved_stage1_isolation_forest.joblib',
  'cascade_tuned_stage1_models_path': '/workspace/models/pump/pump__gold__cascade_stage3_improved_stage1_isolation_forest.joblib',
  '

----

## Finalize the Ledger and Close the Tracking Run

This step writes the cascade ledger to disk and cleanly closes the experiment tracking run.

By the time I get here, the important modeling and artifact creation work is already complete. So this section is mainly about wrapping up the notebook in a structured way.

In [88]:
ledger.add(
    kind="step",
    step="finalize_cascade_modeling",
    message="Gold cascade modeling notebook complete.",
    data={
        "cascade_metrics": cascade_metrics,
        "cascade_results_path_csv": str(CASCADE_RESULTS_PATH_CSV),
        "cascade_results_path_pickle": str(CASCADE_RESULTS_PATH_PICKLE),
        "stage1_model_artifact_path": str(STAGE1_MODEL_ARTIFACT_PATH),
        "stage1_models_path": str(STAGE1_MODELS_PATH),
        "stage2_model_artifact_path": str(STAGE2_MODEL_ARTIFACT_PATH),
        "stage2_models_path": str(STAGE2_MODELS_PATH),
    },
    logger=logger,
)

cascade_ledger_path = GOLD_ARTIFACTS_PATH / GOLD_CASCADE_STAGE3_IMPROVED_LEDGER_FILE_NAME
ledger.write_json(cascade_ledger_path)

wandb.save(str(cascade_ledger_path))
wandb_run.finish()

2026-03-23 23:45:09,550 | INFO | capstone.gold | LEDGER | {'ts_utc': '2026-03-23T23:45:09.550485+00:00', 'stage': 'gold', 'recipe': 'gold_modeling__v001_cascade', 'kind': 'step', 'step': 'finalize_cascade_modeling', 'message': 'Gold cascade modeling notebook complete.', 'why': None, 'consequence': None, 'data': {'cascade_metrics': {'model': '3-Stage Cascade', 'stage1_alert_count_all_rows': 22742, 'stage2_alert_count_all_rows': 13898, 'final_alert_count_all_rows': 13898, 'stage1_alert_count_test_rows': 2288, 'stage2_alert_count_test_rows': 648, 'final_alert_count_test_rows': 648, 'stage3_relaxed_alert_count_test_rows': 648, 'stage3_medium_alert_count_test_rows': 646, 'stage3_strict_alert_count_test_rows': 62, 'precision': 0.09259259259259259, 'recall': 0.5084745762711864, 'f1': 0.1566579634464752, 'stage3_relaxed_precision': 0.09259259259259259, 'stage3_relaxed_recall': 0.5084745762711864, 'stage3_relaxed_f1': 0.1566579634464752, 'stage3_medium_precision': 0.09287925696594428, 'stage3_m

----

## Run Final Lineage and Consistency Checks

Before I treat the tuned cascade notebook as complete, I run a final sanity check on the saved cascade results and truth artifacts.

This check verifies things like:
- required lineage columns exist in the results dataframe
- the dataframe truth hash matches the saved cascade truth
- the parent truth hash matches the Gold parent truth
- the saved truth file exists
- the saved metadata points back to the correct cascade truth hash

I like ending with this because it confirms that the cascade output is not only saved, but also internally consistent and traceable.

In [89]:
required_cascade_meta_columns = [
    "meta__truth_hash",
    "meta__parent_truth_hash",
    "meta__pipeline_mode",
]

missing_cascade_meta_columns = [
    column_name
    for column_name in required_cascade_meta_columns
    if column_name not in cascade_results.columns
]
if missing_cascade_meta_columns:
    raise ValueError(
        f"cascade_results is missing required lineage columns: {missing_cascade_meta_columns}"
    )

cascade_results_truth_hash_check = extract_truth_hash(cascade_results)
if cascade_results_truth_hash_check is None:
    raise ValueError("cascade_results does not contain a readable meta__truth_hash value.")

if cascade_results_truth_hash_check != CASCADE_TRUTH_HASH:
    raise ValueError(
        "cascade_results truth hash does not match CASCADE_TRUTH_HASH:\n"
        f"dataframe={cascade_results_truth_hash_check}\n"
        f"record={CASCADE_TRUTH_HASH}"
    )

cascade_parent_values = cascade_results["meta__parent_truth_hash"].dropna().astype(str).unique().tolist()
if not cascade_parent_values:
    raise ValueError("cascade_results is missing populated meta__parent_truth_hash values.")

if len(cascade_parent_values) != 1:
    raise ValueError(f"cascade_results has multiple parent truth hashes: {cascade_parent_values}")

if cascade_parent_values[0] != GOLD_PARENT_TRUTH_HASH:
    raise ValueError(
        "cascade_results parent truth hash does not match GOLD_PARENT_TRUTH_HASH:\n"
        f"dataframe_parent={cascade_parent_values[0]}\n"
        f"gold_parent={GOLD_PARENT_TRUTH_HASH}"
    )

if not Path(cascade_truth_path).exists():
    raise FileNotFoundError(f"Cascade truth file was not created: {cascade_truth_path}")

loaded_cascade_truth = load_json(cascade_truth_path)

if loaded_cascade_truth.get("truth_hash") != CASCADE_TRUTH_HASH:
    raise ValueError(
        "Saved Cascade truth file hash does not match CASCADE_TRUTH_HASH:\n"
        f"file={loaded_cascade_truth.get('truth_hash')}\n"
        f"record={CASCADE_TRUTH_HASH}"
    )

if loaded_cascade_truth.get("parent_truth_hash") != GOLD_PARENT_TRUTH_HASH:
    raise ValueError(
        "Saved Cascade truth file parent hash does not match GOLD_PARENT_TRUTH_HASH:\n"
        f"truth={loaded_cascade_truth.get('parent_truth_hash')}\n"
        f"gold_parent={GOLD_PARENT_TRUTH_HASH}"
    )

saved_cascade_metadata = load_json(CASCADE_METADATA_PATH)
if saved_cascade_metadata.get("cascade_truth_hash") != CASCADE_TRUTH_HASH:
    raise ValueError(
        "cascade_metadata cascade_truth_hash does not match CASCADE_TRUTH_HASH:\n"
        f"metadata={saved_cascade_metadata.get('cascade_truth_hash')}\n"
        f"record={CASCADE_TRUTH_HASH}"
    )

print("Gold Cascade lineage sanity check passed.")

2026-03-23 23:46:20,293 | INFO | capstone.file_io | Loading JSON: /workspace/artifacts/truths/gold_cascade/pump__gold_cascade__truth__92e0d51aa2536a9417b3721ee4f15c058c43f347bb3a9086aa9239c61ab6a9ca.json
2026-03-23 23:46:20,317 | INFO | capstone.file_io | Loading JSON: /workspace/artifacts/gold/pump/pump__gold__cascade_stage3_improved_metadata.json


Gold Cascade lineage sanity check passed.


### Ask

What does this final sanity check really confirm?

### Answer

It confirms that the tuned cascade results can be trusted as a pipeline artifact, not just as notebook output.

A modeling notebook can run successfully and still leave behind broken lineage or mismatched truth metadata. This final check helps guard against that by making sure the dataframe, truth record, and saved metadata all agree with each other.

So this is really a trust check more than a completion check.

----

## Optional PostgreSQL Write for Tuned Cascade Outputs

This section is for writing the tuned cascade results into PostgreSQL.

I am treating this as an optional persistence step rather than part of the core tuned cascade modeling deliverable. The main outputs from this notebook are still the saved cascade results, model artifacts, thresholds, summary, metadata, ledger, search results, and truth record. Writing to SQL is useful when I want the tuned cascade artifact available for querying, validation, or downstream integration work.

SQL_SCHEMAS = {
    "bronze": "bronze",
    "silver": "silver",
    "gold": "gold",
    "synthetic": "synthetic",
    "truth": "truth",
    "audit": "audit",
}

GOLD_ARTIFACT_TABLES = {
    "baseline_results": "baseline_results",
    "baseline_summary": "baseline_summary",
    "baseline_thresholds": "baseline_thresholds",
    "cascade_results": "cascade_results",
    "cascade_summary": "cascade_summary",
    "cascade_thresholds": "cascade_thresholds",
    "comparison_summary": "comparison_summary",
}

SYNTHETIC_ARTIFACT_TABLES = {
    "batch": "batch",
    "stream": "stream",
    "sensor_messages": "sensor_messages",
}




GOLD_SCHEMA = SQL_SCHEMAS["gold"]

engine = get_engine_from_env()



gold_cascade_results_sql = prepare_layer_dataframe(
    cascade_results,
    truth_hash=CASCADE_TRUTH_HASH,
    parent_truth_hash=GOLD_PARENT_TRUTH_HASH,
    pipeline_mode=PIPELINE_MODE,
    process_run_id=GOLD_PROCESS_RUN_ID,
    add_loaded_at_column=True,
)

gold_cascade_results_table = write_layer_dataframe(
    engine=engine,
    dataframe=gold_cascade_results_sql,
    schema=GOLD_SCHEMA,
    dataset_name=DATASET_NAME,
    artifact_name=GOLD_ARTIFACT_TABLES["cascade_results"],
    if_exists="replace",
    index=False,
)

print(f"Wrote table: {GOLD_SCHEMA}.{gold_cascade_results_table}")